In [ ]:
#| default_exp mcp

## MCP server

Expose nbskill notebook operations as native MCP tools. This is the preferred integration for careful single-notebook reads and edits because multiline notebook cells travel as structured tool arguments rather than shell-quoted strings.

The server now favors verifiable context: batch edits include read-back hashes, symbol graph calls include complete structured usage data, and generated-file warnings are reserved for changes that touch exported notebook code.

The command-line functions are useful on their own, but coding agents work best when the same operations are available as structured tools. This notebook exposes the project through a FastMCP server while keeping the server layer thin and predictable.

### Production contract

The MCP server exposes the production core through stable structured tools. Tool schemas must hide CLI-only flags, responses must include concise text plus structured content, diagnostics must be scoped to touched notebooks when possible, edit tools must report feedback without raw notebook JSON, and experimental tools must be clearly marked or omitted from default production use.

The MCP server should stay boring on purpose. Each tool accepts structured arguments, captures printed output, uses notebook locks where file operations can collide, and delegates the actual work to the same functions tested elsewhere.

```python
mcp = create_mcp()
# MCP clients see tools such as context, write_nb, update_cell, exec_nb, and diff_nb.
```

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import nbskill.mcp as _mcp_mod
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_raw_nb
from nbskill.mcp import capture_call as _example_capture_call
from nbskill.mcp import create_mcp as _example_create_mcp
from nbskill.read import context as _example_context
from nbskill.write import write_nb as _example_write_nb
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
def demo_tool():
    print("captured output")

print(_example_capture_call(demo_tool))
print(type(_example_create_mcp()).__name__)

captured output
FastMCP


In [ ]:
#| export
import asyncio
import json,os
import re
import shutil
import subprocess
import sys
import threading
import time
from contextlib import redirect_stdout, redirect_stderr
from importlib.metadata import PackageNotFoundError, version
from io import StringIO
from pathlib import Path
from urllib.parse import unquote, urlparse

In [ ]:
#| export
from fastcore.nbio import read_nb as _read_raw_nb
from fastcore.nbio import write_nb as _write_raw_nb
from fastmcp import Context, FastMCP
from fastmcp.server.middleware import Middleware
from fastmcp.tools import ToolResult
from mcp.types import TextContent

In [ ]:
#| export
from nbdev.doclinks import nbdev_export as _run_nb_export
from nbskill.convert import convert
from nbskill.edit_interactive import execute_plan
from nbskill.edit_interactive import execute_project_plan
from nbskill.edit_interactive import plan_result_text
from nbskill.execute import exec_nb
from nbskill.workbench import agent_workbench_result
from nbskill.foundation import empty_failure_map, failure_map_path, load_failure_map
from nbskill.foundation import cell_source, exported_py_path, generated_owner, git_root, git_status_paths
from nbskill.foundation import notebook_paths, path_candidates, source_hash, stamp_export_metadata

In [ ]:
#| export
from nbskill.graph import notebook_knowledge_graph_data
from nbskill.graph import notebook_order_problems
from nbskill.graph import private_symbol_report
from nbskill.knowledge import reference_add
from nbskill.knowledge import reference_ingest
from nbskill.knowledge import reference_list
from nbskill.knowledge import reference_query
from nbskill.parallel import notebook_locks
from nbskill.read import context, filter_context

In [ ]:
#| export
from nbskill.edit import edit_notebook
from nbskill.review import reset_global_usage_summary
from nbskill.review import notebook_size_problems
from nbskill.review import notebook_validation_problems
from nbskill.review import public_function_literacy_problems
from nbskill.review import run_style_check
from nbskill.review import style_check
from nbskill.review import style_report
from nbskill.review import diff_nb
from nbskill.review import notebook_autofix
from nbskill.write import should_run_cell_feedback
from nbskill.write import source_lines_cells

### Capturing command output

The MCP tools should return text, not leak stdout and stderr into the server process. These helpers capture each underlying function call and convert its visible result into one response string.

In [ ]:
#| export
def as_text(value):
    return "" if value is None else str(value)

In [ ]:
#| export
_CAPTURE_LOCK = threading.RLock()

In [ ]:
#| export
_REDACT_KEYS = {"new", "new_lines", "source_lines", "replacement_lines", "cells", "edits", "operations", "source", "old_str", "new_str"}

In [ ]:
#| export
_REMOVED_SCRIPT_NAMES = (
    "nbskill-mcp", "read-nb", "write-nb", "update-cell", "batch-edit-nb",
    "show-doc", "exec-nb", "diff-nb", "style-check", "symbol-graph", "private-symbol-report",
)

In [ ]:
#| export
def _package_version(name="nbskill"):
    try: return version(name)
    except PackageNotFoundError: return "unknown"

In [ ]:
#| export
def capture_call(func, **kwargs):
    out, err = StringIO(), StringIO()
    with _CAPTURE_LOCK:
        original_stdout, original_stderr = sys.stdout, sys.stderr
        try:
            try:
                with redirect_stdout(out), redirect_stderr(err):
                    result = func(**kwargs)
            except SystemExit as exc:
                chunks = []
                if out.getvalue(): chunks.append(out.getvalue().rstrip())
                if err.getvalue(): chunks.append(err.getvalue().rstrip())
                chunks.append(f"SystemExit: {exc.code}")
                raise RuntimeError(chr(10).join(chunk for chunk in chunks if chunk)) from exc
        finally:
            sys.stdout, sys.stderr = original_stdout, original_stderr
    chunks = []
    if out.getvalue(): chunks.append(out.getvalue().rstrip())
    if err.getvalue(): chunks.append(err.getvalue().rstrip())
    if result is not None and not chunks: chunks.append(as_text(result))
    return chr(10).join(chunk for chunk in chunks if chunk)

In [ ]:
#| export
def capture_notebook_call(func, *paths, **kwargs):
    "Capture a call while holding per-notebook locks for `paths`."
    with notebook_locks(*paths):
        return capture_call(func, **kwargs)

In [ ]:
#| export
def _mcp_find_cell(path, cell_id):
    nb = _read_raw_nb(path)
    for cell in nb.cells:
        if getattr(cell, "id", None) == cell_id: return cell
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")

In [ ]:
#| export
def _mcp_cell_source_hash(path, cell_id):
    return source_hash(cell_source(_mcp_find_cell(path, cell_id)))

In [ ]:
#| export
def _mcp_expected_hash_warning(path, cell_id, expected_hash):
    if not expected_hash: return None
    actual = _mcp_cell_source_hash(path, cell_id)
    if actual == expected_hash: return None
    return _warning(
        "expected_hash_mismatch",
        f"Skipped edit for {path} id={cell_id}: expected hash {expected_hash}, found {actual}.",
        "Refresh context for the notebook, chapter, or cell and retry with the current source.",
        path=str(path), cell_id=cell_id, expected_hash=expected_hash, actual_hash=actual,
    )

In [ ]:
#| export
def _capture_exec_nb_cli_call(arguments):
    call_args = {key: value for key, value in arguments.items() if key not in {"detail", "workspace_root"}}
    script = "; ".join([
        "import json, sys",
        "from nbskill.execute import exec_nb",
        "exec_nb(**json.loads(sys.argv[1]))",
    ])
    lock_paths = [call_args.get("path")]
    if call_args.get("dest"): lock_paths.append(call_args["dest"])
    cwd = git_root(call_args.get("path") or ".") or _git_base(call_args.get("path") or ".").resolve()
    with notebook_locks(*lock_paths):
        proc = subprocess.run(
            [sys.executable, "-c", script, json.dumps(call_args)],
            text=True, capture_output=True, cwd=str(cwd),
        )
    chunks = [item.rstrip() for item in (proc.stdout, proc.stderr) if item]
    output = chr(10).join(chunks)
    if proc.returncode != 0:
        raise RuntimeError(output or f"exec_nb subprocess failed with exit code {proc.returncode}")
    return output

In [ ]:
#| export
def _mcp_cell_index(path, cell_id):
    nb = _read_raw_nb(path)
    for index, cell in enumerate(nb.cells):
        if getattr(cell, "id", None) == cell_id: return index
    raise ValueError(f"Cell id {cell_id!r} was not found in {path}")

In [ ]:
#| export
def _mcp_cell_ids_from_index(path, start, count):
    if start is None or count <= 0: return []
    nb = _read_raw_nb(path)
    return [
        getattr(cell, "id", None) for cell in nb.cells[start:start + count]
        if getattr(cell, "id", None)
    ]

In [ ]:
#| export
def _mcp_structured_cell_count(cells, default_cell_type="code"):
    return sum(len(source_lines_cells(cell, default_cell_type)) for cell in (cells or []))

In [ ]:
#| export
def _mcp_feedback_location(path, edit, default_cell_type="code"):
    edit_path = str(edit.get("path") or path)
    op = edit.get("op")
    if op == "replace_cell":
        cell_id = edit.get("cell_id")
        return edit_path, _mcp_cell_index(edit_path, cell_id), _mcp_structured_cell_count([edit], default_cell_type)
    if op == "replace_range":
        return edit_path, _mcp_cell_index(edit_path, edit.get("cell_id")), 1
    if op in {"insert_before", "insert_after"}:
        anchor_id = edit.get("anchor_id") or edit.get("cell_id")
        anchor_index = _mcp_cell_index(edit_path, anchor_id)
        start = anchor_index if op == "insert_before" else anchor_index + 1
        return edit_path, start, _mcp_structured_cell_count(edit.get("cells") or [edit], default_cell_type)
    return edit_path, None, 0

In [ ]:
#| export
def _mcp_feedback_output(path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    if not auto_feedback: return ""
    nb = _read_raw_nb(path)
    cells_by_id = {getattr(cell, "id", None): cell for cell in nb.cells}
    target_id = None
    for cell_id in cell_ids:
        cell = cells_by_id.get(cell_id)
        if cell is not None and should_run_cell_feedback(cell): target_id = cell_id
    if target_id is None: return ""
    output = _capture_exec_nb_cli_call(dict(
        path=str(path), dest=None, exc_stop=False, up2id=target_id, chapter=None,
        timeout=feedback_timeout, show_output=True, verbose=False, safe=feedback_safe,
        allow=None, ok_dests=None, cache_httpx=False, cache_dir=None,
        cache_domains=None, allow_new=True, check_only=True,
    ))
    return f"Auto feedback (up to id={target_id}):\n{output}".rstrip()

In [ ]:
#| export
def _append_mcp_feedback(message, path, cell_ids, auto_feedback=True, feedback_timeout=10, feedback_safe=True):
    feedback = _mcp_feedback_output(path, cell_ids, auto_feedback, feedback_timeout, feedback_safe)
    return "\n\n".join(chunk for chunk in [message.rstrip(), feedback] if chunk)

In [ ]:
#| export
def _json_preview(value, limit=1200):
    text = json.dumps(value, indent=2, sort_keys=True, default=str)
    if len(text) <= limit: return text
    return f"{text[:limit].rstrip()}\n... truncated ..."

In [ ]:
#| export
def _text_preview(value, limit=12000):
    text = as_text(value)
    if limit is None or len(text) <= limit:
        return {"text": text, "truncated": False, "chars": len(text), "omitted_chars": 0}
    omitted = len(text) - limit
    return {
        "text": f"{text[:limit].rstrip()}\n... truncated {omitted} chars ...",
        "truncated": True,
        "chars": len(text),
        "omitted_chars": omitted,
    }


def _bounded_structured_value(value, text_limit=12000, list_limit=200, depth=0, max_depth=8):
    if depth >= max_depth: return "<nested data omitted>"
    if isinstance(value, str): return _text_preview(value, limit=text_limit)["text"]
    if isinstance(value, dict):
        return {
            str(key): _bounded_structured_value(val, text_limit=text_limit, list_limit=list_limit, depth=depth + 1, max_depth=max_depth)
            for key, val in value.items()
        }
    if isinstance(value, (list, tuple, set)):
        items = list(value)
        bounded = [
            _bounded_structured_value(item, text_limit=text_limit, list_limit=list_limit, depth=depth + 1, max_depth=max_depth)
            for item in items[:list_limit]
        ]
        if len(items) > list_limit: bounded.append(f"... {len(items) - list_limit} items omitted ...")
        return bounded
    if isinstance(value, Path): return str(value)
    return value


def _mcp_clamp_int(value, default, minimum, maximum, name):
    if value is None: value = default
    try:
        value = int(value)
    except (TypeError, ValueError):
        raise ValueError(f"{name} must be an integer") from None
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}")
    return min(value, maximum)


def _mcp_clamp_float(value, default, minimum, maximum, name):
    if value is None: value = default
    try:
        value = float(value)
    except (TypeError, ValueError):
        raise ValueError(f"{name} must be a number") from None
    if value < minimum:
        raise ValueError(f"{name} must be >= {minimum}")
    return min(value, maximum)

In [ ]:
#| export
def _redact_value(key, value, limit=160):
    if value is None: return None
    text = as_text(value)
    if key in _REDACT_KEYS and len(text) > limit:
        return f"<{len(text)} chars redacted; use detail='debug' to inspect>"
    if len(text) > limit * 3:
        return f"{text[:limit].rstrip()}... <{len(text) - limit} more chars>"
    return value

In [ ]:
#| export
def _redact_arguments(arguments):
    return {key: _redact_value(key, value) for key, value in (arguments or {}).items()}

In [ ]:
#| export
_MCP_LOG_ENV = "NBSKILL_MCP_LOG"
_MCP_LOG_LOCK = threading.RLock()
_MCP_LOG_DISABLED = {"", "0", "false", "off", "none"}


def _mcp_log_path():
    raw = os.environ.get(_MCP_LOG_ENV)
    if raw is not None and raw.strip().lower() in _MCP_LOG_DISABLED: return None
    return Path(raw).expanduser() if raw else Path.home() / ".nbskill-mcp.jsonl"


def _mcp_json_safe(value):
    if isinstance(value, dict): return {str(key): _mcp_json_safe(_redact_value(str(key), val)) for key, val in value.items()}
    if isinstance(value, (list, tuple, set)): return [_mcp_json_safe(item) for item in value]
    if isinstance(value, Path): return str(value)
    if isinstance(value, (str, int, float, bool)) or value is None: return value
    return as_text(value)


def _mcp_log_event(event, **data):
    path = _mcp_log_path()
    if path is None: return
    payload = {
        "ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "pid": os.getpid(),
        "event": event,
    }
    payload.update({key: _mcp_json_safe(value) for key, value in data.items()})
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        with _MCP_LOG_LOCK, path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(payload, sort_keys=True) + "\n")
    except Exception:
        pass


def _mcp_log_exception(event, exc, **data):
    _mcp_log_event(event, error_type=type(exc).__name__, error=as_text(exc), **data)

In [ ]:
#| export
def _mcp_middleware_tool_name(context):
    return getattr(getattr(context, "message", None), "name", "")


def _mcp_middleware_tool_arguments(context):
    return getattr(getattr(context, "message", None), "arguments", None) or {}


class _NbskillMCPLogMiddleware(Middleware):
    async def on_initialize(self, context, call_next):
        _mcp_log_event("initialize_start", method=context.method)
        try: result = await call_next(context)
        except BaseException as exc:
            _mcp_log_exception("initialize_error", exc, method=context.method)
            raise
        _mcp_log_event("initialize_end", method=context.method)
        return result

    async def on_list_tools(self, context, call_next):
        _mcp_log_event("tools_list_start", method=context.method)
        try: result = await call_next(context)
        except BaseException as exc:
            _mcp_log_exception("tools_list_error", exc, method=context.method)
            raise
        _mcp_log_event("tools_list_end", method=context.method, count=len(result or []))
        return result

    async def on_call_tool(self, context, call_next):
        tool = _mcp_middleware_tool_name(context)
        arguments = _mcp_middleware_tool_arguments(context)
        started = time.monotonic()
        _mcp_log_event("tool_start", tool=tool, arguments=arguments)
        try: result = await call_next(context)
        except BaseException as exc:
            elapsed_ms = int((time.monotonic() - started) * 1000)
            _mcp_log_exception("tool_error", exc, tool=tool, arguments=arguments, elapsed_ms=elapsed_ms)
            raise
        elapsed_ms = int((time.monotonic() - started) * 1000)
        _mcp_log_event("tool_end", tool=tool, elapsed_ms=elapsed_ms)
        return result

In [ ]:
#| export
def _warning(code, message, next_action=None, **extra):
    item = {"code": code, "message": message}
    if next_action: item["next_action"] = next_action
    item.update({key: value for key, value in extra.items() if value is not None})
    return item

In [ ]:
#| export
def _path_without_cwd_prefix(path):
    return path_candidates(path)[-1]

In [ ]:
#| export
def _git_base(path="."):
    base = _path_without_cwd_prefix(path)
    if (base.exists() and not base.is_dir()) or (not base.exists() and base.suffix):
        return base.parent
    return base

In [ ]:
#| export
def _rooted_path(path, root):
    raw = Path(str(path)).expanduser()
    pth = _path_without_cwd_prefix(path)
    candidates = [pth]
    if root is not None and not raw.is_absolute():
        candidates = [Path(root) / raw, Path(root) / pth, *candidates]
    for candidate in candidates:
        try:
            if candidate.exists(): return candidate.resolve()
        except OSError:
            continue
    return pth.resolve()

In [ ]:
#| export
def _file_uri_path(uri):
    parsed = urlparse(str(uri))
    if parsed.scheme != "file": return None
    netloc = f"//{parsed.netloc}" if parsed.netloc else ""
    return Path(unquote(f"{netloc}{parsed.path}"))


async def _mcp_client_roots(ctx=None):
    if ctx is None: return []
    try: roots = await ctx.list_roots()
    except Exception: return []
    paths = []
    for root in roots:
        path = _file_uri_path(getattr(root, "uri", ""))
        if path is not None: paths.append(path)
    return paths


async def _mcp_workspace_root(ctx=None):
    for path in await _mcp_client_roots(ctx):
        try:
            if path.exists(): return path.resolve()
        except OSError:
            continue
    return None


def _mcp_workspace_path(path, root):
    if path in (None, ""): return path
    raw = Path(str(path)).expanduser()
    if raw.is_absolute() or root is None: return str(raw)
    return str((Path(root) / raw).resolve())


def _mcp_workspace_paths(paths, root):
    if paths in (None, ""): return paths
    return ",".join(_mcp_workspace_path(item.strip(), root) for item in str(paths).split(",") if item.strip())


def _mcp_workspace_edit_paths(edits, root):
    resolved = []
    for edit in edits:
        item = dict(edit)
        if item.get("path"): item["path"] = _mcp_workspace_path(item["path"], root)
        resolved.append(item)
    return resolved


def _mcp_workspace_call_args(arguments, root, *keys):
    data = dict(arguments)
    for key in keys:
        if key in data: data[key] = _mcp_workspace_path(data[key], root)
    return data

In [ ]:
#| export
def _rel_to_root(path, root):
    try: return _rooted_path(path, root).relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError): return str(path)

In [ ]:
#| export
def _generated_files(root):
    skip = {".git", ".venv", "__pycache__", ".mypy_cache", ".pytest_cache"}
    items = []
    for path in Path(root).rglob("*.py"):
        if any(part in skip for part in path.parts): continue
        owner = generated_owner(path)
        if owner is not None: items.append((path, owner))
    return items

In [ ]:
#| export
def _owner_output(path):
    owner = generated_owner(path)
    if owner is None: return f"No generated-notebook owner found for {path}"
    return f"Generated file owner: {path} -> {owner}"

In [ ]:
#| export
def _failure_data():
    path = failure_map_path()
    try: return load_failure_map(path) if path.exists() else empty_failure_map()
    except OSError: return empty_failure_map()

In [ ]:
#| export
def _resolve_diagnostic_scope(path, root, scope_path=None):
    raw = path if scope_path in (None, "") else scope_path
    if raw in (None, "", "."): return Path(root)
    scoped = Path(raw).expanduser()
    if not scoped.is_absolute(): scoped = Path(root) / scoped
    return scoped

In [ ]:
#| export
def _is_project_scope(path, root):
    try: return Path(path).resolve() == Path(root).resolve()
    except OSError: return False

In [ ]:
#| export
def _generated_pairs_for_scope(root, path):
    path = Path(path)
    if _is_project_scope(path, root): return _generated_files(root)
    if path.suffix == ".py":
        owner = generated_owner(path)
        return [(path, owner)] if owner is not None else []
    pairs = []
    for nb_path in notebook_paths(path):
        try: py_path = exported_py_path(nb_path)
        except (FileNotFoundError, OSError): py_path = None
        if py_path is not None and Path(py_path).exists(): pairs.append((Path(py_path), nb_path.resolve()))
    return pairs

In [ ]:
#| export
def _repair_export_hash_metadata(path="."):
    root = git_root(path) or _git_base(path).resolve()
    diagnostic_path = _resolve_diagnostic_scope(path, root)
    repairs = []
    repair_codes = {"exported-py-hash-mismatch", "missing-exported-py-hash"}
    for problem in notebook_validation_problems(diagnostic_path):
        if problem.get("code") not in repair_codes: continue
        nb_path = Path(problem["path"])
        with notebook_locks(nb_path):
            nb = _read_raw_nb(nb_path)
            py_path = exported_py_path(nb_path, nb)
            if py_path is None: continue
            _run_nb_export(path=str(nb_path))
            if not py_path.exists(): continue
            nb = _read_raw_nb(nb_path)
            stamp_export_metadata(nb, py_path)
            _write_raw_nb(nb, nb_path)
        repairs.append(dict(path=str(nb_path), generated=str(py_path), code=problem.get("code")))
    return repairs

In [ ]:
#| export
def _cell_source_text(cell):
    if isinstance(cell, dict):
        source = cell.get("source", "")
        return "".join(source) if isinstance(source, list) else str(source)
    return cell_source(cell)

In [ ]:
#| export
def _cell_type_text(cell):
    return cell.get("cell_type", "") if isinstance(cell, dict) else getattr(cell, "cell_type", "")

In [ ]:
#| export
def _cell_export_relevant(cell):
    if _cell_type_text(cell) != "code": return False
    return any(re.match(r"^\s*#\|\s*(export|default_exp)\b", line) for line in _cell_source_text(cell).splitlines())

In [ ]:
#| export
def _notebook_export_relevant_change(owner, root):
    owner = Path(owner)
    root = Path(root)
    owner_rel = _rel_to_root(owner, root)
    proc = subprocess.run(["git", "-C", str(root), "show", f"HEAD:{owner_rel}"], text=True, capture_output=True)
    if proc.returncode != 0: return True
    try:
        old_nb = json.loads(proc.stdout)
        new_nb = json.loads(owner.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return True
    old_cells = {cell.get("id"): cell for cell in old_nb.get("cells", []) if cell.get("id")}
    new_cells = {cell.get("id"): cell for cell in new_nb.get("cells", []) if cell.get("id")}
    for cell_id in set(old_cells) | set(new_cells):
        old_cell, new_cell = old_cells.get(cell_id), new_cells.get(cell_id)
        if old_cell is None or new_cell is None:
            changed = True
        else:
            changed = _cell_type_text(old_cell) != _cell_type_text(new_cell) or _cell_source_text(old_cell) != _cell_source_text(new_cell)
        if changed and any(_cell_export_relevant(cell) for cell in (old_cell, new_cell) if cell is not None):
            return True
    return False

In [ ]:
#| export
def _style_problem_warnings(path, root):
    warnings = []
    problems = [*notebook_size_problems(path), *public_function_literacy_problems(path)]
    for problem in problems:
        code = problem.get("code")
        if code == "large-cell":
            cell = f" cell id={problem['cell_id']}" if problem.get("cell_id") else ""
            warnings.append(_warning(
                "large_cell",
                f"Notebook {_rel_to_root(problem.get('path'), root)}{cell} is large: {problem.get('detail')}.",
                "Split the cell so it contains one idea before continuing.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
        elif code == "large-generated-py":
            generated = problem.get("exported_py_path")
            warnings.append(_warning(
                "large_generated_py",
                f"Generated file {_rel_to_root(generated, root)} is getting large: {problem.get('detail')}.",
                "Use the notebook split tool to split the source notebook/module.",
                path=problem.get("path"), generated=generated, problem=problem,
            ))
        elif str(code).startswith("public-function-"):
            missing = problem.get("missing") or "one-line docstring"
            symbol = problem.get("symbol", "<unknown>")
            warnings.append(_warning(
                code.replace("-", "_"),
                f"Notebook {_rel_to_root(problem.get('path'), root)} public function {symbol!r} is missing {missing}.",
                "Add Markdown docs, a one-line docstring, an example cell, and a focused test cell.",
                path=problem.get("path"), cell_id=problem.get("cell_id"), problem=problem,
            ))
    return warnings

In [ ]:
#| export
def _warning_scope_root_rel(path, root):
    if path in (None, ""): return ""
    try:
        return Path(path).expanduser().resolve().relative_to(Path(root).resolve()).as_posix()
    except (OSError, ValueError):
        return str(path).strip()

In [ ]:
#| export
def _warning_scope_matches(value, root, scope):
    if scope is None: return True
    rel = _warning_scope_root_rel(value, root)
    return any(rel == item or rel.startswith(f"{item.rstrip('/')}/") for item in scope)

In [ ]:
#| export
def _warning_related_paths(item):
    for key in ("path", "generated", "owner"):
        if item.get(key): yield item[key]
    problem = item.get("problem") or {}
    for key in ("path", "exported_py_path"):
        if problem.get(key): yield problem[key]

In [ ]:
#| export
def _warning_cell_id(item):
    problem = item.get("problem") or {}
    return item.get("cell_id") or problem.get("cell_id")

In [ ]:
#| export
_NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES = {
    "generated_without_notebook", "notebook_export_missing", "exported_py_hash_mismatch",
}

In [ ]:
#| export
def _filter_warnings_for_scope(warnings, root, scope_path=None, scope_cell_ids=None):
    if scope_path in (None, "", ".") and scope_cell_ids is None: return warnings
    scope = None if scope_path in (None, "", ".") else {_warning_scope_root_rel(scope_path, root).rstrip("/")}
    cell_filter_active = scope_cell_ids is not None
    cell_ids = set(scope_cell_ids or [])
    filtered = []
    for item in warnings:
        if scope is not None and not any(_warning_scope_matches(path, root, scope) for path in _warning_related_paths(item)):
            continue
        cell_id = _warning_cell_id(item)
        if cell_filter_active and cell_id and cell_id not in cell_ids: continue
        if cell_filter_active and not cell_id and item.get("code") not in _NOTEBOOK_LEVEL_CONTEXT_WARNING_CODES: continue
        filtered.append(item)
    return filtered

In [ ]:
#| export
def _doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
    root = git_root(path) or _git_base(path).resolve()
    diagnostic_path = _resolve_diagnostic_scope(path, root, scope_path)
    project_scope = _is_project_scope(diagnostic_path, root)
    changed = git_status_paths(root) if (root / ".git").exists() else set()
    warnings = []
    for py_path, owner in _generated_pairs_for_scope(root, diagnostic_path):
        py_rel = _rel_to_root(py_path, root)
        owner_rel = _rel_to_root(owner, root)
        if py_rel in changed and owner_rel not in changed:
            warnings.append(_warning(
                "generated_without_notebook",
                f"Generated file {py_rel} changed without its source notebook {owner_rel}.",
                "Move the edit into the notebook and export, or verify the generated edit is intentional.",
                path=py_rel, owner=owner_rel,
            ))
        if owner_rel in changed and py_rel not in changed:
            if _notebook_export_relevant_change(owner, root):
                warnings.append(_warning(
                    "notebook_export_missing",
                    f"Notebook {owner_rel} changed but generated file {py_rel} is unchanged.",
                    "Run export or use an nbskill write tool before shipping.",
                    path=owner_rel, generated=py_rel,
                ))
    for problem in notebook_validation_problems(diagnostic_path):
        if problem.get("code") != "exported-py-hash-mismatch": continue
        warnings.append(_warning(
            "exported_py_hash_mismatch",
            f"Notebook {problem['path']} metadata does not match current generated file {problem.get('exported_py_path')}.",
            "Run nbskill_validate or export the notebook through nbskill write tools.",
            path=problem.get("path"), generated=problem.get("exported_py_path"),
        ))
    if project_scope:
        warnings.extend(_doc_script_warnings(root))
        failures = _failure_data().get("events", [])[-10:]
        recent_failures = [event for event in failures if event.get("kind") == "failure"]
        if recent_failures:
            last = recent_failures[-1]
            warnings.append(_warning(
                "recent_tool_failures",
                f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
                "Run doctor(detail='debug') or style_check(delete_after_output=True) after resolving it.",
                tool=last.get("tool"),
            ))
    if scope_path is None: scope_path = path
    return _filter_warnings_for_scope(warnings, root, scope_path, scope_cell_ids)

In [ ]:
#| export
def _doc_script_warnings(root):
    docs = [Path(root) / "README.md", Path(root) / "nbskill" / "SKILL.md"]
    docs += list((Path(root) / "nbskill" / "references").glob("*.md")) if (Path(root) / "nbskill" / "references").exists() else []
    warnings = []
    for doc in docs:
        if not doc.exists(): continue
        try: text = doc.read_text(encoding="utf-8", errors="ignore")
        except OSError: continue
        found = sorted(name for name in _REMOVED_SCRIPT_NAMES if name in text)
        if found:
            warnings.append(_warning(
                "removed_script_name",
                f"{doc.relative_to(root)} references removed CLI names: {', '.join(found)}.",
                "Replace hyphenated command names with underscore script names.",
                path=str(doc), names=found,
            ))
    return warnings

In [ ]:
#| export
def _first_match(pattern, text, flags=0):
    match = re.search(pattern, text or "", flags)
    return match.group(1) if match else None

In [ ]:
#| export
def _line_cell_ids(text):
    return set(re.findall(r"^Cell id=([^\s:]+)", text or "", re.MULTILINE))

In [ ]:
#| export
def _response_scope_cell_ids(tool, arguments, preview):
    text = preview.get("text", "")
    if tool == "context":
        ids = set(re.findall(r"\bCell id=([^\s:]+)", text))
        return ids or None
    if tool == "diff_nb":
        ids = set(re.findall(r"--- code cell ([^\s]+) ---", text))
        return ids if ids else set()
    if tool == "edit_notebook":
        ids = arguments.get("affected_cell_ids") or []
        if not ids and isinstance(arguments.get("edit_notebook"), dict):
            ids = arguments["edit_notebook"].get("affected_cell_ids", [])
        if not ids:
            ids = [str(item.get("cell_id") or item.get("anchor_id")) for item in arguments.get("edits", []) if isinstance(item, dict) and (item.get("cell_id") or item.get("anchor_id"))]
        return set(str(item) for item in ids if item) or None
    if tool == "exec_nb":
        up2id = arguments.get("up2id")
        if isinstance(up2id, str) and up2id and not up2id.isdigit(): return {up2id}
        return _line_cell_ids(text) or None
    return None

In [ ]:
#| export
def _unique_strings(items):
    seen, result = set(), []
    for item in items:
        if item in seen: continue
        seen.add(item)
        result.append(item)
    return result

In [ ]:
#| export
def _notebook_paths_from_text(text):
    return _unique_strings(re.findall(r"(?<![\w.-])(?:[~./A-Za-z0-9_-]+\.ipynb)", text or ""))

In [ ]:
#| export
def _edit_scope_paths(arguments):
    default = arguments.get("path")
    edits = arguments.get("edits") or []
    paths = [default] if default else []
    paths += [item.get("path") for item in edits if isinstance(item, dict) and item.get("path")]
    return _unique_strings(str(item) for item in paths if item)

In [ ]:
#| export
def _response_scope_paths(tool, arguments, preview):
    text_paths = _notebook_paths_from_text(preview.get("text", ""))
    if tool == "edit_notebook": return _unique_strings([*_edit_scope_paths(arguments), *text_paths])
    path = arguments.get("path") or arguments.get("nb_path")
    return [str(path)] if path else []

In [ ]:
#| export
def _dedupe_warnings(warnings):
    seen, result = set(), []
    for item in warnings:
        key = json.dumps(item, sort_keys=True, default=str)
        if key in seen: continue
        seen.add(key)
        result.append(item)
    return result

In [ ]:
#| export
def _response_warnings(tool, arguments, preview):
    warnings = []
    if preview.get("truncated"):
        warnings.append(_warning(
            "output_truncated",
            f"{tool} output was truncated by {preview['omitted_chars']} chars.",
            "Repeat with a narrower query or detail='debug' if you need full context.",
        ))
    read_context_tools = {"context"}
    scoped_project_tools = {"exec_nb"}
    if tool in read_context_tools or tool == "edit_notebook": return _dedupe_warnings(warnings)
    if tool == "diff_nb":
        path = arguments.get("path") or "."
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids)[:3])
    elif tool in scoped_project_tools:
        cell_ids = _response_scope_cell_ids(tool, arguments, preview)
        for path in _response_scope_paths(tool, arguments, preview):
            warnings.extend(_doctor_warnings(path, scope_path=path, scope_cell_ids=cell_ids))
    return _dedupe_warnings(warnings)[:3]

In [ ]:
#| export
def _brief_call(tool, arguments, preview, status="completed"):
    lines = [f"{tool} {status}"]
    keys = (
        "path", "target", "scope", "notebook", "notebooks", "cell_id", "id",
        "chapter", "name", "any_cell_id", "symbol", "action", "query", "mode",
    )
    for key in keys:
        value = arguments.get(key)
        if value not in (None, ""): lines.append(f"{key}={value}")
    if arguments.get("dry_run") is True: lines.append("dry_run=True")
    if preview.get("truncated"): lines.append(f"output_truncated=True omitted_chars={preview['omitted_chars']}")
    return lines

In [ ]:
#| export
def _first_nonblank(text):
    for line in as_text(text).splitlines():
        if line.strip(): return line.strip()
    return ""


def _short_item(text, limit=120):
    text = " ".join(_first_nonblank(text).split())
    return text if len(text) <= limit else text[:limit - 3].rstrip() + "..."


def _definition_items(text, limit=120):
    items = []
    for line in as_text(text).splitlines():
        stripped = line.strip()
        if not re.match(r"^(async\s+def|def|class)\s+", stripped): continue
        items.append(stripped if len(stripped) <= limit else stripped[:limit - 3].rstrip() + "...")
    return items


def _warning_summary_lines(warnings, limit=5):
    lines = []
    for item in (warnings or [])[:limit]:
        message = item.get("message") or item.get("detail") or str(item)
        next_action = item.get("next_action")
        suffix = f" Next: {next_action}" if next_action else ""
        lines.append(f"- {message}{suffix}")
    return lines


def _diagnostic_line(item):
    path = item.get("path") or ""
    cell = f" id={item.get('cell_id')}" if item.get("cell_id") else ""
    code = item.get("code") or item.get("severity") or "diagnostic"
    detail = item.get("detail") or item.get("message") or ""
    where = f"{path}{cell}".strip()
    return f"- {code}: {where} {detail}".rstrip()


def _top_code_counts(chart, limit=5):
    by_code = (chart or {}).get("by_code", {})
    return ", ".join(f"{code}={count}" for code, count in list(by_code.items())[:limit])


def _context_summary_lines(data):
    data = data or {}
    selection = data.get("selection") or {}
    cells = data.get("cells") or selection.get("cells") or []
    notebooks = data.get("notebooks") or selection.get("notebooks") or []
    symbols = data.get("symbols") or selection.get("symbols") or []
    lines = [f"resolved={data.get('resolved_kind') or data.get('kind')}"]
    if data.get("path"): lines.append(f"path={data['path']}")
    if data.get("symbol"): lines.append(f"symbol={data['symbol']}")
    if notebooks: lines.append(f"notebooks={len(notebooks)}")
    if cells: lines.append(f"cells={len(cells)}")
    if symbols: lines.append(f"symbols={len(symbols)}")
    for cell in cells[:3]:
        label = f"{cell.get('path', data.get('path', ''))} id={cell.get('cell_id', '')}".strip()
        lines.append(f"- {label}: {_short_item(cell.get('source', ''))}")
    return lines


def _filter_context_match_summary(item):
    label = f"{item.get('path')} id={item.get('cell_id')} idx={item.get('cell_idx')}"
    definitions = _definition_items(item.get("source", ""))
    if not definitions: return [f"- {label}: {_short_item(item.get('source', ''))}"]
    return [f"- {label}:", *[f"  {line}" for line in definitions]]


def _filter_context_summary_lines(data):
    data = data or {}
    matches = data.get("matches", [])
    lines = [f"matches={len(matches)} shown of {data.get('total_matches', len(matches))}"]
    for item in matches[:5]: lines.extend(_filter_context_match_summary(item))
    return lines


def _doctor_summary_lines(report):
    report = report or {}
    errors, warnings = report.get("errors", []), report.get("warnings", [])
    lines = [f"errors={len(errors)} warnings={len(warnings)}"]
    issues = [*errors, *warnings] or report.get("issues", [])
    lines.extend(_diagnostic_line(item) for item in issues[:5])
    return lines


def _style_summary_lines(report):
    report = report or {}
    summary = report.get("summary", {})
    lines = [
        f"diagnostics={summary.get('diagnostic_count', 0)}",
        f"notebook_problems={summary.get('notebook_problem_count', 0)}",
        f"chkstyle_problems={summary.get('chkstyle_problem_count', 0)}",
    ]
    counts = _top_code_counts(report.get("problem_chart"))
    if counts: lines.append(f"top_codes={counts}")
    diagnostics = report.get("diagnostics") or report.get("notebook_problems") or []
    lines.extend(_diagnostic_line(item) for item in diagnostics[:5])
    return lines


def _diff_summary_lines(text):
    text = as_text(text)
    cells = re.findall(r"--- code cell ([^\s-]+) ---", text)
    added = sum(1 for line in text.splitlines() if line.startswith("+") and not line.startswith("+++"))
    deleted = sum(1 for line in text.splitlines() if line.startswith("-") and not line.startswith("---"))
    lines = [f"changed_cells={len(cells)} added_lines={added} deleted_lines={deleted}"]
    if cells: lines.append("cells=" + ", ".join(cells[:8]))
    first = _first_nonblank(text)
    if not cells and first: lines.append(_short_item(first))
    return lines


def _reference_summary_lines(data):
    data = data or {}
    hits = data.get("hits", [])
    lines = [f"hits={len(hits)} backend={data.get('backend', '')}".rstrip()]
    for hit in hits[:5]:
        label = ".".join(item for item in [hit.get("module"), hit.get("symbol")] if item)
        dep = hit.get("dependency_status")
        score = hit.get("score")
        score_text = f" score={score:.2f}" if isinstance(score, (int, float)) else ""
        lines.append(f"- {label or hit.get('path')}: {dep}{score_text}")
    return lines


def _edit_summary_lines(data):
    data = data or {}
    diffs = [item for item in data.get("diffs", []) if item.get("changed")]
    affected = data.get("affected_cell_ids") or []
    lines = [
        f"changed={data.get('changed')}",
        f"dry_run={data.get('dry_run', False)}",
        f"affected_cells={len(affected)}",
        f"changed_ops={len(diffs)}",
    ]
    if affected: lines.append("cells=" + ", ".join(affected[:8]))
    return lines


def _exec_summary_lines(text):
    text = as_text(text)
    outputs = re.findall(r"--- output id=([^\s]+) ---", text)
    lines = [f"outputs={len(outputs)}"]
    if outputs: lines.append("ids=" + ", ".join(outputs[:8]))
    first = _first_nonblank(text)
    if first: lines.append(_short_item(first))
    return lines


def _workbench_summary_lines(data):
    data = data or {}
    contract = data.get("contract") or {}
    context_data = data.get("context") or {}
    notebooks = context_data.get("selected_notebooks") or []
    lines = [f"goal={_short_item(contract.get('goal') or data.get('summary') or '')}"]
    if notebooks: lines.append("notebooks=" + ", ".join(item.get("path", "") for item in notebooks[:5]))
    if data.get("expected_gates"): lines.append("gates=" + ", ".join(data["expected_gates"].get("hard", [])[:5]))
    return [line for line in lines if line and not line.endswith("=")]


def _tool_summary_lines(tool, arguments, full_text, preview, status, structured, warnings):
    lines = _brief_call(tool, arguments or {}, preview, status=status)
    if tool == "context": lines.extend(_context_summary_lines(structured.get("context")))
    elif tool == "filter_context": lines.extend(_filter_context_summary_lines(structured.get("filter_context")))
    elif tool == "doctor": lines.extend(_doctor_summary_lines(structured.get("doctor")))
    elif tool == "style_check": lines.extend(_style_summary_lines(structured.get("style_report")))
    elif tool == "diff_nb": lines.extend(_diff_summary_lines(full_text))
    elif tool == "reference": lines.extend(_reference_summary_lines(structured.get("reference")))
    elif tool == "edit_notebook": lines.extend(_edit_summary_lines(structured.get("edit_notebook") or structured))
    elif tool == "exec_nb": lines.extend(_exec_summary_lines(full_text))
    elif tool == "agent_workbench": lines.extend(_workbench_summary_lines(structured.get("agent_workbench")))
    else:
        first = _first_nonblank(full_text)
        if first and status != "completed": lines.append(_short_item(first))
    if warnings:
        lines += ["", "Warnings:"]
        lines.extend(_warning_summary_lines(warnings))
    return lines


def mcp_tool_result(tool, arguments, full_output, max_output_chars=12000, detail="summary", warnings=None, hints=None, status="completed", **structured):
    "Return concise visible MCP text plus structured data for clients that inspect it."
    detail = detail or "summary"
    if max_output_chars is None:
        max_output_chars = 50000 if detail == "debug" else 12000
    if detail in {"debug", "full"} and max_output_chars is not None:
        max_output_chars = max(max_output_chars, 50000) if detail == "debug" else max_output_chars
    full_text = as_text(full_output or "")
    preview = _text_preview(full_text, limit=max_output_chars)
    structured_text_limit = 50000 if detail == "debug" else 12000
    auto_warnings = [] if status != "completed" else _response_warnings(tool, arguments or {}, preview)
    warnings = _dedupe_warnings([*(warnings or []), *auto_warnings])
    hints = list(hints or [])
    lines = _tool_summary_lines(tool, arguments or {}, full_text, preview, status, structured, warnings)
    if detail in {"debug", "full"} and preview["text"]: lines += ["", "Result:", preview["text"]]
    if detail == "debug" and hints:
        lines += ["", "Hints:"]
        lines.extend(f"- {hint}" for hint in hints)
    summary = chr(10).join(lines)
    data = {
        "summary": summary,
        "status": status,
        "call": {"tool": tool, "arguments": _redact_arguments(arguments or {})},
        "full_output": preview["text"],
        "preview_output": preview["text"],
        "output_truncated": preview["truncated"],
        "output_chars": preview["chars"],
        "omitted_chars": preview["omitted_chars"],
        "warnings": warnings,
        "hints": hints if detail == "debug" else [],
    }
    if detail == "debug": data["debug"] = {"arguments": arguments or {}, "raw_output": _text_preview(full_text, limit=structured_text_limit)["text"]}
    for key, value in structured.items():
        data["result_summary" if key == "summary" else key] = _bounded_structured_value(value, text_limit=structured_text_limit)
    return ToolResult(content=[TextContent(type="text", text=summary)], structured_content=data)


def _mcp_tool_error_result(tool, arguments, exc, max_output_chars=12000, detail="summary", **structured):
    "Return an MCP result for a tool failure without raising through the transport."
    message = f"{tool} failed: {type(exc).__name__}: {exc}"
    warning = _warning(
        "tool_failed", message,
        "Treat this call as failed, but the MCP transport remains usable; fix the input or run a narrower diagnostic.",
    )
    return mcp_tool_result(
        tool, arguments, message, max_output_chars=max_output_chars, detail=detail,
        warnings=[warning], status="failed", ok=False,
        error={"type": type(exc).__name__, "message": str(exc)}, **structured,
    )

In [ ]:
#| export
def _status_data(client_roots=None):
    scripts = [
        "context", "write_nb", "update_cell", "batch_edit_nb", "exec_nb", "diff_nb", "style_check",
        "install_nbskill", "agent_workbench", "convert", "reference",
        "nbskill_mcp", "nbskill_mcp_start", "nbskill_mcp_stop", "nbskill_mcp_restart",
    ]
    client_roots = [str(Path(item).resolve()) for item in (client_roots or [])]
    workspace_root = client_roots[0] if client_roots else str(Path.cwd())
    try: version_text = version("nbskill")
    except PackageNotFoundError: version_text = "unknown"
    command = "nbskill_mcp"
    log_path = _mcp_log_path()
    return {
        "version": version_text,
        "cwd": str(Path.cwd()),
        "workspace_root": workspace_root,
        "client_roots": client_roots,
        "python": sys.executable,
        "pid": os.getpid(),
        "mcp_command": command,
        "mcp_command_path": shutil.which(command),
        "mcp_log_path": str(log_path) if log_path is not None else None,
        "cli_tools": {name: shutil.which(name) for name in scripts},
        "reconnect_hint": "Restart or reconnect the MCP client after reinstalling nbskill or changing tool signatures.",
        "install_commands": [
            "uv tool install --editable . --force",
            "codex mcp add nbskill -- nbskill_mcp",
            "claude mcp add nbskill -- nbskill_mcp",
            "install_nbskill --target cursor --cursor_workspace .",
        ],
    }

In [ ]:
#| export
def _doctor_fix_line(item, root):
    path = _rel_to_root(item.get("path"), root)
    if item.get("cell_id"):
        return f"- {path} cell {item['cell_id']}: {item.get('description', item.get('code', 'fixed'))}"
    if item.get("generated"): return f"- refreshed export hash for {path}"
    return f"- {path}: {item.get('description', item.get('code', 'fixed'))}"

In [ ]:
#| export
def _doctor_report(
    path=".",
    detail="summary",
    fix=False,
    reset=False,
    scopes="error,warning",
    skip_folder_re=None,
    skip_path=None,
    max_output_chars=12000,
    max_diagnostics=200,
):
    root = git_root(path) or _git_base(path).resolve()
    selected = _doctor_scope_set(scopes)
    export_fixes = _repair_export_hash_metadata(path) if fix else []
    style_fixes = notebook_autofix(path) if fix and "style" in selected else []
    fixes = [*export_fixes, *style_fixes]
    status = _status_data()
    errors = _doctor_error_items(path, status) if "error" in selected else []
    warnings, private_text = _doctor_warning_items(path) if "warning" in selected else ([], "")
    style = (
        _doctor_style_report(path, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
        if "style" in selected else None
    )
    failure_map = _failure_data()
    if reset: reset_global_usage_summary()
    hints = [
        "Use scopes='error,warning,style' or scopes='all' for the full doctor report.",
        "Use scopes='style' to include chkstyle output; chkstyle is omitted from error/warning scopes.",
        "Use context, then edit_notebook for deterministic notebook mutations.",
    ]
    changed = sorted(git_status_paths(root)) if (root / ".git").exists() else []
    generated = [
        {"path": _rel_to_root(py, root), "owner": _rel_to_root(owner, root)}
        for py, owner in _generated_files(root)
    ]
    issue_count = len(errors) + len(warnings)
    style_count = (style or {}).get("summary", {}).get("diagnostic_count", 0)
    summary = f"nbskill doctor: {len(errors)} error(s), {len(warnings)} warning(s)"
    if "style" in selected: summary += f", {style_count} style diagnostic(s)"
    if not issue_count and "style" not in selected: summary = "nbskill doctor: no actionable errors or warnings"
    text_lines = [summary]
    if fixes:
        text_lines.append("\nFixes:")
        text_lines.extend(_doctor_fix_line(item, root) for item in fixes)
    if errors:
        text_lines.append("\nErrors:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in errors)
    if warnings:
        text_lines.append("\nWarnings:")
        text_lines.extend(f"- {item['message']}" + (f" Next: {item['next_action']}" if item.get("next_action") else "") for item in warnings)
    if style is not None and style.get("text", "").strip():
        text_lines.append("\nStyle:")
        text_lines.append(style["text"].strip())
    report = {
        "path": str(path),
        "root": str(root),
        "scopes": sorted(selected),
        "status": status,
        "errors": errors,
        "warnings": warnings,
        "issues": [*errors, *warnings],
        "style": style,
        "private_symbol_report": private_text if "warning" in selected else "",
        "hints": hints,
        "capabilities": [item for item in _MCP_CAPABILITIES.split(",") if item],
        "changed_paths": changed if detail == "debug" else changed[:20],
        "generated_owners": generated if detail == "debug" else generated[:20],
        "recent_events": failure_map.get("events", [])[-20:] if detail == "debug" else [],
        "counts": failure_map.get("counts", {}),
        "reset": bool(reset),
        "fix": {"requested": bool(fix), "applied": fixes},
        "text": "\n".join(text_lines),
    }
    return report

In [ ]:
#| export
def nbskill_status(json_output: bool = False):  # Keep argument for compatibility with the CLI wrapper
    "Report nbskill version, MCP command setup, canonical CLI tools, and reconnect hints."
    return _status_data()

In [ ]:
#| export
_DOCTOR_SCOPES = {"error", "warning", "style"}

In [ ]:
#| export
def _doctor_scope_set(scopes="error,warning"):
    "Normalize comma/space-separated doctor scopes."
    if scopes is None: return {"error", "warning"}
    raw = str(scopes).replace(",", " ").split()
    selected = set(raw) or {"error", "warning"}
    if "all" in selected: return set(_DOCTOR_SCOPES)
    unknown = selected - _DOCTOR_SCOPES
    if unknown: raise ValueError(f"Unknown doctor scope(s): {', '.join(sorted(unknown))}")
    return selected

In [ ]:
#| export
def _problem_message(problem):
    parts = [str(problem.get("path") or "")]
    if problem.get("cell_id"): parts.append(f"id={problem['cell_id']}")
    if problem.get("line"): parts.append(f"line={problem['line']}")
    if problem.get("symbol"): parts.append(f"symbol={problem['symbol']!r}")
    if problem.get("code"): parts.append(f"code={problem['code']}")
    if problem.get("detail"): parts.append(str(problem["detail"]))
    return " ".join(part for part in parts if part)

In [ ]:
#| export
def _doctor_validation_errors(path):
    errors = []
    for problem in notebook_validation_problems(path):
        errors.append(_warning(
            problem.get("code", "notebook-validation"),
            _problem_message(problem),
            "Fix notebook metadata/source ordering before relying on notebook edits.",
            severity="error", scope="error", problem=problem,
        ))
    return errors

In [ ]:
#| export
def _doctor_order_errors(path):
    errors = []
    try:
        problems = notebook_order_problems(path)
    except FileNotFoundError as exc:
        return [_warning(
            "notebook_order_missing_file",
            f"Notebook order scan found a missing notebook: {exc.filename or exc}",
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="error", scope="error", path=exc.filename,
        )]
    root = git_root(path) or _git_base(path).resolve()
    diagnostic_path = _resolve_diagnostic_scope(path, root)
    project_scope = _is_project_scope(diagnostic_path, root)
    for problem in problems:
        if project_scope and Path(problem.get("path", "")).name == "manage.ipynb": continue
        errors.append(_warning(
            problem.get("code", "notebook-order"),
            _problem_message(problem),
            "Move definitions/imports before use, or add the missing import.",
            severity="error", scope="error", problem=problem,
        ))
    return errors

In [ ]:
#| export
def _doctor_recent_failure_errors():
    errors = []
    recent = [event for event in _failure_data().get("events", [])[-10:] if event.get("kind") == "failure"]
    if recent:
        last = recent[-1]
        errors.append(_warning(
            "recent_tool_failures",
            f"Recent nbskill failure: {last.get('tool')} {last.get('summary') or last.get('error')}",
            "Run doctor(detail='debug', scopes='error') after resolving it.",
            severity="error", scope="error", tool=last.get("tool"),
        ))
    return errors

In [ ]:
#| export
def _doctor_error_items(path, status):
    errors = [
        *_doctor_validation_errors(path),
        *_doctor_order_errors(path),
        *_doctor_recent_failure_errors(),
    ]
    if not status.get("mcp_command_path"):
        errors.append(_warning(
            "mcp_command_missing",
            "nbskill_mcp is not on PATH for this process.",
            "Run uv tool install --editable . --force and reconnect the MCP client.",
            severity="error", scope="error",
        ))
    return errors

In [ ]:
#| export
def _private_symbol_report_text(path):
    return capture_call(private_symbol_report, path=str(path))

In [ ]:
#| export
def _private_symbol_warnings(path):
    try:
        text = _private_symbol_report_text(path)
    except FileNotFoundError as exc:
        text = f"Private symbol scan found a missing notebook: {exc.filename or exc}"
        return [_warning(
            "private_symbol_missing_file",
            text,
            "Remove stale notebook references or recreate the missing notebook before rerunning doctor.",
            severity="warning", scope="warning", path=exc.filename,
        )], text
    if "No cross-notebook private symbol calls found." in text:
        return [], text
    warnings = [
        _warning(
            "private_symbol_call",
            line[2:],
            "Promote the helper to public API or keep the call inside the defining notebook.",
            severity="warning", scope="warning",
        )
        for line in text.splitlines()
        if line.startswith("- ")
    ]
    return warnings, text

In [ ]:
#| export
def _doctor_warning_items(path):
    error_codes = {"exported_py_hash_mismatch", "recent_tool_failures"}
    warnings = [
        {**item, "severity": item.get("severity", "warning"), "scope": "warning"}
        for item in _doctor_warnings(path)
        if item.get("code") not in error_codes
    ]
    private_warnings, private_text = _private_symbol_warnings(path)
    return [*warnings, *private_warnings], private_text

In [ ]:
#| export
def _doctor_style_report(path, skip_folder_re=None, skip_path=None, max_output_chars=12000, max_diagnostics=200):
    chkstyle = run_style_check(path, skip_folder_re, skip_path, strict=False, max_output_chars=max_output_chars)
    try:
        return style_report(
            path, chkstyle=chkstyle, max_output_chars=max_output_chars,
            max_diagnostics=max_diagnostics, skip_folder_re=skip_folder_re, skip_path=skip_path,
        )
    except FileNotFoundError as exc:
        message = f"Style scan found a missing notebook: {exc.filename or exc}"
        diagnostic = {
            "source": "notebook",
            "code": "style_missing_file",
            "severity": "error",
            "path": exc.filename,
            "detail": message,
        }
        output = chkstyle.get("output", "") if isinstance(chkstyle, dict) else ""
        truncated = max_output_chars is not None and len(output) > max_output_chars
        text = output[:max_output_chars].rstrip() if truncated else output.strip()
        if text: text = f"{text}\n\n{message}"
        else: text = message
        return {
            "path": str(path),
            "summary": {
                "notebook_problem_count": 1,
                "chkstyle_problem_count": 0,
                "diagnostic_count": 1,
                "recent_problem_count": 0,
                "output_truncated": truncated,
                "output_chars": len(output),
                "omitted_chars": max(0, len(output) - (max_output_chars or len(output))),
            },
            "diagnostics": [diagnostic],
            "problem_chart": {
                "by_code": {"style_missing_file": 1},
                "by_severity": {"error": 1},
                "by_path": {str(exc.filename): 1} if exc.filename else {},
                "by_source": {"notebook": 1},
            },
            "notebook_problems": [diagnostic],
            "global_usage": {},
            "chkstyle": {"status": chkstyle.get("status", 0) if isinstance(chkstyle, dict) else 0, "text": output[:max_output_chars] if truncated else output, "truncated": truncated, "chars": len(output), "omitted_chars": max(0, len(output) - (max_output_chars or len(output)))},
            "fixes": [],
            "text": text,
        }

In [ ]:
data = _status_data()
assert data["mcp_command"] == "nbskill_mcp"
assert "batch_edit_nb" in data["cli_tools"]

### Registering notebook tools

`create_mcp` is the bridge between this package and an agent client. Each tool is a thin wrapper around a public function, with notebook locks around operations that touch shared files.

The wrapper should preserve useful structure, not flatten everything into text. `context` uses `overview` for compact versus focused detail, and project-level calls can attach the notebook knowledge graph with `include_graph=True`. Cell or symbol targets include symbol graph payloads so MCP clients can inspect impact without parsing prose.

Edit and execution wrappers also adapt to MCP-specific transport. `update_cell` accepts `new_lines` so exact source can be sent directly without temporary files, and unsafe `exec_nb` runs through a subprocess because the underlying timeout machinery needs the main interpreter thread.

In [ ]:
#| export
_MCP_DIAGNOSTIC_TOOL_CATALOG = {
    'healthcheck': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'setup'),
        'description': 'Cheap liveness probe for the nbskill MCP server, installed version, capabilities, concurrency policy, and reconnect hints.',
        'when_to_use': 'Call first when checking that the MCP server is connected or after reinstalling/exporting tool signatures.',
        'combine_with': 'Could be folded into doctor, but a cheap health probe is useful enough to keep separate.',
    },
    'doctor': {
        'feature': 'diagnostics',
        'usefulness': 'core',
        'tags': ('status', 'diagnostics', 'error', 'warning', 'style'),
        'description': 'Scoped diagnostics for MCP setup, fatal notebook problems, warnings, private symbol leaks, and optional chkstyle output.',
        'when_to_use': "Use scopes='error', scopes='warning', scopes='style', or scopes='all' depending on the diagnostic depth needed.",
        'combine_with': 'Now absorbs private symbol warnings and scoped style diagnostics; healthcheck stays separate as a cheap probe.',
    },
}

In [ ]:
#| export
_MCP_READ_CONTEXT_TOOL_CATALOG = {
    'context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'context', 'project', 'notebook', 'cell', 'symbol', 'graph'),
        'description': 'Single notebook-aware reader for project, notebook, chapter, cell id, or Python symbol targets, with optional project graph context.',
        'when_to_use': "Use first: pass target='project', a notebook path/name, chapter title, cell id, or Python symbol; add include_graph=True with target='project' when placement, reuse, or structure advice needs the notebook knowledge graph.",
        'combine_with': 'Project graph context is embedded with include_graph=True; cell and symbol targets include symbol graph payloads, so no separate symbol graph tool is needed.',
    },
    'filter_context': {
        'feature': 'read_context',
        'usefulness': 'core',
        'tags': ('read', 'context', 'project', 'notebook', 'search', 'filter'),
        'description': 'Project-wide notebook-aware context search that filters cells by query selectors and include/exclude regexes.',
        'when_to_use': 'Use when you need matching cells across a project or notebook scope instead of one resolved target.',
        'combine_with': 'Use context on a returned path/cell id when a match needs deeper local rationale or symbol impact.',
    },
}

In [ ]:
#| export
_MCP_READ_DOC_TOOL_CATALOG = {}


In [ ]:
#| export
_MCP_CELL_EDIT_TOOL_CATALOG = {
    'edit_notebook': {
        'feature': 'notebook_edit',
        'usefulness': 'core',
        'tags': ('edit', 'notebook', 'cell', 'text', 'batch'),
        'description': 'Apply deterministic notebook edit operations atomically across cells, lines, and notebook-wide text replacements; returns structured match counts, hashes, affected cell ids, diffs, warnings, and optional feedback.',
        'when_to_use': "Use after context identifies target cells. Use replace_text/replace_texts with target='all' for notebook-level renames, line ops for focused cell edits, and structural ops for insert/delete/move/replace cell changes.",
        'combine_with': 'Read context before editing; review with diff_nb, exec_nb(check_only=True), doctor, or style_check afterwards.',
    },
}

In [ ]:
#| export
_MCP_BATCH_EDIT_TOOL_CATALOG = {}

In [ ]:
#| export
_MCP_REVIEW_TOOL_CATALOG = {
    'exec_nb': {
        'feature': 'verification',
        'usefulness': 'core',
        'tags': ('execute', 'notebook', 'verify', 'safe'),
        'description': 'Execute a notebook, chapter, or cells up to an id with safe-mode controls and visible output/error capture; unsafe execution is routed through a CLI subprocess so signal-based timeouts run in a main interpreter without blocking MCP worker threads.',
        'when_to_use': 'Use after edits or before trusting notebook behavior; use check_only=True when outputs should not be written.',
        'combine_with': 'Keep separate because execution has distinct safety and concurrency semantics.',
    },
    'diff_nb': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'notebook', 'diff'),
        'description': 'Notebook-aware code-cell diff that avoids raw .ipynb noise and can map generated Python diffs back to notebook owners.',
        'when_to_use': 'Use before final reporting or when reviewing notebook edits without expanding JSON metadata churn.',
        'combine_with': 'Could be grouped with style_check under review, but diff parameters and output are meaningfully different.',
    },
    'style_check': {
        'feature': 'review',
        'usefulness': 'core',
        'tags': ('review', 'style', 'hygiene', 'privacy'),
        'description': 'Notebook hygiene and style report including chkstyle output, private symbol warnings, duplicate imports, and order issues.',
        'when_to_use': 'Use after substantial edits or when a notebook feels structurally messy.',
        'combine_with': "Doctor can include style diagnostics with scopes='style'; standalone style_check remains the explicit review tool.",
    },
}

In [ ]:
#| export
_MCP_AGENT_TOOL_CATALOG = {
    'execute_plan': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'edit', 'plan', 'notebook', 'project'),
        'description': 'Run a bounded edit-interactive plan against one notebook or a project-scoped set of notebooks.',
        'when_to_use': "Use scope='notebook' with notebook=... for one notebook, or scope='project' with notebooks=... for broad plans.",
        'combine_with': 'Combined former execute_project_plan into this tool via scope.',
    },
    'agent_workbench': {
        'feature': 'agentic_planning',
        'usefulness': 'advanced',
        'tags': ('agent', 'context', 'taste', 'contract', 'review'),
        'description': 'Prepare or execute a taste-aware small-diff workbench run with context, budgets, and gates.',
        'when_to_use': 'Use before autonomous implementation when taste, scope, context, and patch budgets need to be explicit.',
        'combine_with': 'Sits above execute_plan; execute_plan remains the bounded notebook executor.',
    },
}

In [ ]:
#| export
_MCP_KNOWLEDGE_TOOL_CATALOG = {
    'reference': {
        'feature': 'reference_knowledge',
        'usefulness': 'core',
        'tags': ('knowledge', 'reference', 'search', 'implementation'),
        'description': 'Manage and search globally indexed reference implementations with add, list, ingest, and query actions.',
        'when_to_use': 'Use query before implementing similar code; use add/list/ingest to maintain the local reference index.',
        'combine_with': 'Combines add, list, ingest, and query as actions on one tool.',
    },
}

In [ ]:
#| export
_MCP_CONVERT_TOOL_CATALOG = {
    'convert': {
        'feature': 'conversion',
        'usefulness': 'situational',
        'tags': ('convert', 'python', 'notebook', 'folder', 'project'),
        'description': 'Convert one Python file, a folder of Python files, or a pure-Python package into nbdev notebooks/project structure.',
        'when_to_use': 'Use mode="notebook" for file/folder migration and mode="project" for a whole nbdev project scaffold.',
        'combine_with': 'Combines single-file, folder, and project conversion modes.',
    },
}

In [ ]:
#| export
_MCP_TOOL_CATALOG = {
    **_MCP_DIAGNOSTIC_TOOL_CATALOG,
    **_MCP_READ_CONTEXT_TOOL_CATALOG,
    **_MCP_READ_DOC_TOOL_CATALOG,
    **_MCP_CELL_EDIT_TOOL_CATALOG,
    **_MCP_BATCH_EDIT_TOOL_CATALOG,
    **_MCP_REVIEW_TOOL_CATALOG,
    **_MCP_AGENT_TOOL_CATALOG,
    **_MCP_KNOWLEDGE_TOOL_CATALOG,
    **_MCP_CONVERT_TOOL_CATALOG,
}

In [ ]:
#| export
def _mcp_tool_meta(name):
    "Return FastMCP registration metadata for one nbskill tool."
    info = _MCP_TOOL_CATALOG[name]
    return {
        "name": name,
        "description": info["description"],
        "tags": set(info["tags"]),
        "meta": {
            "feature": info["feature"],
            "usefulness": info["usefulness"],
            "when_to_use": info["when_to_use"],
            "combine_with": info["combine_with"],
        },
    }

In [ ]:
#| export
_MCP_CAPABILITIES = ",".join(_MCP_TOOL_CATALOG)
mcp = FastMCP(
    "nbskill",
    instructions=(
        "Work notebook-first in nbdev projects. Feature areas are diagnostics, focused context, "
        "notebook edits, verification/review, symbol impact, agentic planning, reference search, "
        "and conversion. For reading, use context with a project, notebook, chapter title, cell id, "
        "or public symbol target; overview controls compact notebook summaries or implementation focus. "
        "For edits, use edit_notebook as the single production mutation tool. It supports whole-cell, "
        "line-range, insert/delete/move, and notebook-wide replace_text/replace_texts operations with "
        "expected_hash guards and structured diffs. Edit tools default to auto_feedback=True, which runs "
        "only exploratory or test cells in check-only mode and returns their output/errors. Use exec_nb, "
        "diff_nb, and style_check for verification and review; use doctor with scopes='error', 'warning', "
        "'style', or 'all' for diagnostics. Chkstyle output only appears when doctor includes the style "
        "scope. Use reference for indexed reference implementations. Normal tool output is concise; use "
        "detail='debug' only when troubleshooting. Notebook operations are concurrency-safe: calls touching "
        "the same notebook are serialized, calls touching different notebooks can run in parallel, and "
        "execution uses a global semaphore. Keep documentation before exported code and show-off examples "
        "after it."
    ),
)
mcp.add_middleware(_NbskillMCPLogMiddleware())

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("healthcheck"))
async def _healthcheck_tool(detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Return lightweight nbskill MCP status and point deeper diagnostics to doctor."
    client_roots = await _mcp_client_roots(ctx)
    data = _status_data(client_roots)
    full_output = "\n".join([
        "nbskill mcp ok",
        f"version={data['version']}",
        f"cwd={Path.cwd()}",
        f"workspace_root={data['workspace_root']}",
        f"python={sys.executable}",
        f"pid={os.getpid()}",
        f"log_path={data['mcp_log_path']}",
        f"capabilities={_MCP_CAPABILITIES}",
        "parallel=same-notebook operations serialized; different notebooks may run in parallel",
        "execution=global semaphore with one active safe notebook execution",
        "diagnostics=run doctor(scopes='error,warning') for fatal problems and warnings; add style for chkstyle",
        "schema_refresh=restart or reconnect the MCP client after reinstall/export to refresh tool schemas",
    ])
    return mcp_tool_result(
        "healthcheck", {"detail": detail}, full_output, detail=detail,
        status_data=data, capabilities=_MCP_CAPABILITIES.split(","),
    )

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("doctor"))
async def doctor_tool(
    path: str = ".", scopes: str = "error,warning", detail: str = "summary",
    fix: bool = False, reset: bool = False, skip_folder_re: str | None = None,
    skip_path: str | None = None, max_output_chars: int = 12000,
    max_diagnostics: int = 200, ctx: Context = None,
) -> ToolResult:
    "Report scoped MCP diagnostics: errors, warnings, and optional chkstyle/style details."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    skip_path = _mcp_workspace_path(skip_path, root) if skip_path else skip_path
    arguments = dict(path=path, scopes=scopes, detail=detail, fix=fix, reset=reset, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, workspace_root=str(root) if root else None)
    try:
        report = _doctor_report(path=path, detail=detail, fix=fix, reset=reset, scopes=scopes, skip_folder_re=skip_folder_re, skip_path=skip_path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics)
    except Exception as exc:
        return _mcp_tool_error_result("doctor", arguments, exc, max_output_chars=max_output_chars, detail=detail)
    return mcp_tool_result(
        "doctor", arguments, report["text"], detail=detail,
        warnings=report["issues"], hints=report["hints"], doctor=report,
    )

In [ ]:
#| export
def _mcp_graph_summary(graph):
    project = graph.get("project", {})
    lines = [
        f"Notebook knowledge graph: {project.get('notebook_count', 0)} notebook(s), {len(graph.get('nodes', []))} node(s), {len(graph.get('edges', []))} edge(s)"
    ]
    if graph.get("issues"):
        lines.append(f"Issues: {len(graph['issues'])}")
    lines.append("Layers:")
    for layer in graph.get("layers", [])[:8]:
        lines.append(f"- {layer.get('name')}: {len(layer.get('node_ids', []))} node(s)")
    lines.append("Tour:")
    for item in graph.get("tour", [])[:8]:
        lines.append(f"- {item.get('order')}. {item.get('title')}: {len(item.get('node_ids', []))} node(s)")
    return "\n".join(lines)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("context"))
async def _context_tool(
    target: str = "project",
    scope: str = ".",
    overview: bool = False,
    include_graph: bool = False,
    detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Resolve one target to project, notebook, chapter, cell, or symbol context."
    root = await _mcp_workspace_root(ctx)
    scope = _mcp_workspace_path(scope, root)
    target_text = str(target or "")
    target_is_project = target_text in {"project", ".", ""}
    target_is_path = target_text.endswith(".ipynb") or "/" in target_text or "\\" in target_text
    if target_is_path: target = _mcp_workspace_path(target, root)
    arguments = dict(
        target=target,
        scope=scope,
        overview=overview,
        include_graph=include_graph,
        detail=detail,
        workspace_root=str(root) if root else None,
    )
    context_args = dict(target=target, scope=scope, overview=overview)
    try:
        with notebook_locks(scope), _CAPTURE_LOCK:
            out, err = StringIO(), StringIO()
            with redirect_stdout(out), redirect_stderr(err): data = context(**context_args)
    except Exception as exc:
        return _mcp_tool_error_result("context", arguments, exc, detail=detail)

    warnings = []
    graph = None
    if include_graph and target_is_project:
        try:
            with notebook_locks(scope):
                graph = notebook_knowledge_graph_data(scope)
        except Exception as exc:
            warnings.append(_warning(
                "knowledge_graph_failed",
                f"Notebook knowledge graph failed: {type(exc).__name__}: {exc}",
                "Call notebook_knowledge_graph locally or narrow the context scope.",
                scope=str(scope),
            ))
    elif include_graph:
        warnings.append(_warning(
            "knowledge_graph_requires_project",
            "Knowledge graph context is only attached for project-level context targets.",
            "Call context with target='project' and include_graph=True.",
        ))

    full_output = data["text"]
    structured = {"context": data}
    if graph is not None:
        full_output = "\n\n".join([full_output, "Project graph:", _mcp_graph_summary(graph)])
        structured["knowledge_graph"] = graph

    return mcp_tool_result(
        "context",
        arguments,
        full_output,
        detail=detail,
        warnings=warnings,
        **structured,
    )

In [ ]:
#| hide
demo_graph = {
    "project": {"notebook_count": 1},
    "nodes": [{"id": "n"}],
    "edges": [{"source": "n", "target": "n"}],
    "layers": [{"name": "Demo", "node_ids": ["n"]}],
    "tour": [{"order": 0, "title": "Demo", "node_ids": ["n"]}],
    "issues": [],
}
demo_summary = _mcp_graph_summary(demo_graph)
assert "Notebook knowledge graph: 1 notebook(s), 1 node(s), 1 edge(s)" in demo_summary
assert "- Demo: 1 node(s)" in demo_summary

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("filter_context"))
async def _filter_context_tool(
    scope: str = ".",
    query: str | None = None,
    include_re: str | None = None,
    exclude_re: str | None = None,
    max_matches: int = 50,
    max_chars_per_cell: int = 1200,
    verbose: bool = True,
    detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Search notebook cells across a scope with query and regex filters."
    root = await _mcp_workspace_root(ctx)
    scope = _mcp_workspace_path(scope, root)
    max_matches = _mcp_clamp_int(max_matches, 50, 1, 200, "max_matches")
    max_chars_per_cell = _mcp_clamp_int(max_chars_per_cell, 1200, 100, 4000, "max_chars_per_cell")
    arguments = dict(
        scope=scope, query=query, include_re=include_re, exclude_re=exclude_re,
        max_matches=max_matches, max_chars_per_cell=max_chars_per_cell,
        verbose=verbose, detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        with notebook_locks(scope), _CAPTURE_LOCK:
            out, err = StringIO(), StringIO()
            with redirect_stdout(out), redirect_stderr(err):
                data = filter_context(
                    scope=scope, query=query, include_re=include_re, exclude_re=exclude_re,
                    max_matches=max_matches, max_chars_per_cell=max_chars_per_cell,
                    verbose=verbose,
                )
    except Exception as exc:
        return _mcp_tool_error_result("filter_context", arguments, exc, detail=detail)
    return mcp_tool_result("filter_context", arguments, data["text"], detail=detail, filter_context=data)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("edit_notebook"))
async def edit_notebook_tool(
    path: str, edits: list[dict], validate_code: bool = True,
    default_cell_type: str = "code", auto_feedback: bool = True,
    feedback_timeout: int = 10, feedback_safe: bool = True, dry_run: bool = False,
    tool_timeout: float = 150.0, detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Apply deterministic notebook edit operations atomically."
    if not edits: raise ValueError("Pass at least one edit")
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    edits = _mcp_workspace_edit_paths(edits, root)
    tool_timeout = _mcp_clamp_float(tool_timeout, 150.0, 0.001, 150.0, "tool_timeout")
    feedback_timeout = _mcp_clamp_int(feedback_timeout, 10, 1, 60, "feedback_timeout")
    arguments = dict(
        path=path, edits=edits, validate_code=validate_code, default_cell_type=default_cell_type,
        auto_feedback=auto_feedback, feedback_timeout=feedback_timeout, feedback_safe=feedback_safe,
        dry_run=dry_run, tool_timeout=tool_timeout, detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        result = await asyncio.wait_for(
            asyncio.to_thread(
                edit_notebook,
                path, edits, validate_code=validate_code, default_cell_type=default_cell_type,
                auto_feedback=auto_feedback, feedback_timeout=feedback_timeout,
                feedback_safe=feedback_safe, detail=detail, dry_run=dry_run,
            ),
            timeout=tool_timeout,
        )
    except TimeoutError as exc:
        message = f"edit_notebook exceeded its MCP timeout of {tool_timeout:g}s before Codex's client timeout."
        return _mcp_tool_error_result("edit_notebook", arguments, TimeoutError(message), detail=detail)
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("edit_notebook", arguments, exc, detail=detail)
    warnings = result.get("warnings", []) if isinstance(result, dict) else []
    full_output = result.get("text", str(result)) if isinstance(result, dict) else str(result)
    return mcp_tool_result(
        "edit_notebook", arguments, full_output, detail=detail, warnings=warnings,
        ok=bool(result.get("ok", False)) if isinstance(result, dict) else True,
        changed=result.get("changed") if isinstance(result, dict) else None,
        no_change=result.get("no_change") if isinstance(result, dict) else None,
        dry_run=result.get("dry_run") if isinstance(result, dict) else dry_run,
        affected_cell_ids=result.get("affected_cell_ids", []) if isinstance(result, dict) else [],
        before_hash=result.get("before_hash") if isinstance(result, dict) else None,
        after_hash=result.get("after_hash") if isinstance(result, dict) else None,
        planned_hash=result.get("planned_hash") if isinstance(result, dict) else None,
        exported=result.get("exported") if isinstance(result, dict) else None,
        edit_notebook=result if isinstance(result, dict) else {},
    )

In [ ]:
#| export
def _exec_approval_request(path, full_output):
    "Return structured approval details for a safe-exec unapproved-cell block."
    text = as_text(full_output)
    match = re.search(r"unapproved cell id=([A-Za-z0-9_-]+)", text)
    cell_id = match.group(1) if match else None
    request = dict(kind="safe_execution", path=str(path), cell_id=cell_id, source_hash=None, reason="Safe execution refused an unapproved notebook cell.", approval_argument={"allow_new": True}, risk="Runs notebook code that has not yet been stamped by nbskill safe execution.")
    if cell_id:
        try: request["source_hash"] = _mcp_cell_source_hash(path, cell_id)
        except Exception as exc: request["source_hash_error"] = f"{type(exc).__name__}: {exc}"
    return request

In [ ]:
#| export
async def _elicit_exec_approval(ctx, request):
    "Ask the MCP client for permission to rerun safe execution with allow_new=True."
    if ctx is None or not hasattr(ctx, "elicit"):
        return False, dict(action="unavailable", reason="MCP context does not support elicitation.")
    cell = f" cell {request['cell_id']}" if request.get("cell_id") else " an unapproved cell"
    source_hash = f" source hash {request['source_hash']}" if request.get("source_hash") else ""
    message = (
        f"nbskill safe-mode blocked{cell} in {request['path']}{source_hash}. "
        "Approve one rerun with allow_new=True? Only approve if you trust this notebook source."
    )
    try:
        result = await ctx.elicit(
            message, bool, response_title="Approve notebook execution",
            response_description="Allow nbskill to rerun once with allow_new=True for this blocked cell.")
    except Exception as exc: return False, dict(action="error", error_type=type(exc).__name__, message=str(exc))
    action = getattr(result, "action", None)
    if action != "accept": return False, dict(action=action or "unknown")
    approved = bool(getattr(result, "data", False))
    return approved, dict(action="accept", approved=approved)

`exec_nb_tool` runs notebooks through safe execution. When safe-mode refuses a fresh cell, the tool can ask the MCP client for explicit approval before a one-time `allow_new=True` rerun.

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("exec_nb"))
async def exec_nb_tool(
    path: str,
    up2id: int | str | None = None,
    chapter: str | None = None,
    timeout: int = 30,
    show_output: bool = True,
    allow_new: bool = False,
    check_only: bool = False,
    detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Execute a notebook and return visible outputs/errors. Use check_only=True to avoid writing outputs."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(
        path=path, up2id=up2id, chapter=chapter, timeout=timeout, show_output=show_output,
        allow_new=allow_new, check_only=check_only, detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        call_args = dict(
            path=path, dest=None, exc_stop=False, up2id=up2id, chapter=chapter, timeout=timeout,
            show_output=show_output, verbose=False, safe=True, allow=None, ok_dests=None,
            cache_httpx=False, cache_dir=None, cache_domains=None, allow_new=allow_new, check_only=check_only,
        )
        full_output = capture_notebook_call(exec_nb, path, **call_args)
        warnings = []
        approval_request = None
        elicitation = None
        if "PermissionError: Audit:" in full_output:
            warnings.append(_warning(
                "safe-exec-audit-block",
                "Safe execution blocked an audited operation.",
                "Use an explicit CLI run for trusted notebooks that need broader execution permissions.",
            ))
        approval_blocked = "_ExecutionApprovalRequired" in full_output or "refusing to execute unapproved cell" in full_output
        if approval_blocked and not allow_new:
            approval_request = _exec_approval_request(path, full_output)
            approved, elicitation = await _elicit_exec_approval(ctx, approval_request)
            approval_request["elicited"] = True
            approval_request["approved"] = approved
            if approved:
                retry_args = dict(call_args, allow_new=True)
                full_output = capture_notebook_call(exec_nb, path, **retry_args)
                arguments["allow_new"] = True
                arguments["elicited_allow_new"] = True
                approval_blocked = "_ExecutionApprovalRequired" in full_output or "refusing to execute unapproved cell" in full_output
            else:
                warnings.append(_warning(
                    "safe-exec-approval-required",
                    "Safe execution refused an unapproved notebook cell.",
                    "Approve the MCP elicitation or rerun with allow_new=True only if you approve the notebook source.",
                    **approval_request,
                ))
                if elicitation and elicitation.get("action") == "error":
                    warnings.append(_warning(
                        "mcp-elicitation-unavailable",
                        "The MCP client did not complete the execution approval prompt.",
                        "Ask the user in chat before rerunning with allow_new=True.",
                        **elicitation,
                    ))
        if approval_blocked and allow_new:
            warnings.append(_warning(
                "safe-exec-approval-required",
                "Safe execution refused an unapproved notebook cell.",
                "Rerun with allow_new=True only if you approve the notebook source.",
            ))
        return mcp_tool_result(
            "exec_nb", arguments, full_output, detail=detail, warnings=warnings,
            approval_required=approval_request, elicitation=elicitation,
        )
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("exec_nb", arguments, exc, detail=detail)

In [ ]:
#| eval: false
await exec_nb_tool("nbs/data/demo.ipynb", check_only=True)

In [ ]:
#| hide
assert exec_nb_tool.__name__ == "exec_nb_tool"

In [ ]:
#| hide
class _FakeApprovalCtx:
    def __init__(self, data=True):
        self.data = data
        self.calls = []

    async def elicit(self, message, response_type, **kwargs):
        self.calls.append((message, response_type, kwargs))
        return type("Accepted", (), dict(action="accept", data=self.data))()


with write_demo_notebook("07_mcp_exec_approval.ipynb", cells=[mk_cell("print('needs approval')")]) as approval_nb:
    blocked_text = "nbskill: refusing to execute unapproved cell id=" + _read_raw_nb(approval_nb).cells[0].id
    request = _exec_approval_request(approval_nb, blocked_text)
    fake = _FakeApprovalCtx(data=True)
    approved, elicitation = await _elicit_exec_approval(fake, request)
    checks = [request["cell_id"] == _read_raw_nb(approval_nb).cells[0].id, request["source_hash"] == _mcp_cell_source_hash(approval_nb, request["cell_id"]), (approved, elicitation) == (True, {"action": "accept", "approved": True})]
    assert all(checks)
    assert fake.calls and fake.calls[0][1] is bool

In [ ]:
#| hide
request = dict(path="demo.ipynb", cell_id="abc", source_hash="123")
approved, elicitation = await _elicit_exec_approval(None, request)
assert approved is False
assert elicitation["action"] == "unavailable"

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("diff_nb"))
async def _diff_nb_tool(path: str, ref_a: str | None = "HEAD", ref_b: str | None = None, adds: bool = True, changes: bool = True, dels: bool = False, cell_id: str | None = None, after_id: str | None = None, show_owner: bool = False, detail: str = "summary", ctx: Context = None) -> ToolResult:
    "Diff notebook code cells without expanding raw notebook JSON; optionally filter by cell id or map generated Python to its owner."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    arguments = dict(path=path, ref_a=ref_a, ref_b=ref_b, adds=adds, changes=changes, dels=dels, cell_id=cell_id, after_id=after_id, show_owner=show_owner, detail=detail, workspace_root=str(root) if root else None)
    try:
        if show_owner and Path(path).suffix == ".py":
            return mcp_tool_result("diff_nb", arguments, _owner_output(path), detail=detail)
        full_output = capture_notebook_call(diff_nb, path, **{k: v for k, v in arguments.items() if k not in {"show_owner", "detail", "workspace_root"}})
    except Exception as exc:
        return _mcp_tool_error_result("diff_nb", arguments, exc, detail=detail)
    return mcp_tool_result("diff_nb", arguments, full_output, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("execute_plan"))
async def execute_plan_tool(
    plan: str, notebook: str | None = None, scope: str = "notebook",
    notebooks: str | None = None, model: str | None = None, max_steps: int = 8,
    timeout: int = 30, symbols: str | None = None, dry_run: bool = True,
    tool_timeout: float = 150.0, detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Run a bounded notebook-editing subagent against one notebook or a project notebook set."
    root = await _mcp_workspace_root(ctx)
    notebook = _mcp_workspace_path(notebook, root) if notebook else notebook
    notebooks = _mcp_workspace_paths(notebooks, root) if notebooks else notebooks
    max_steps = _mcp_clamp_int(max_steps, 8, 1, 20, "max_steps")
    timeout = _mcp_clamp_int(timeout, 30, 1, 120, "timeout")
    tool_timeout = _mcp_clamp_float(tool_timeout, 150.0, 0.001, 150.0, "tool_timeout")
    arguments = dict(
        plan=plan, notebook=notebook, scope=scope, notebooks=notebooks, model=model, max_steps=max_steps,
        timeout=timeout, symbols=symbols, dry_run=dry_run, tool_timeout=tool_timeout,
        detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        mode = "project" if scope == "project" or notebooks else "notebook"
        if mode == "project":
            project_args = dict(
                plan=plan, notebooks=notebooks, model=model, max_steps=max_steps, timeout=timeout,
                dry_run=dry_run, symbols=symbols,
            )
            full_output = await asyncio.wait_for(
                asyncio.to_thread(capture_call, execute_project_plan, **project_args),
                timeout=tool_timeout,
            )
            return mcp_tool_result("execute_plan", arguments, full_output, detail=detail)
        if not notebook: raise ValueError("notebook is required when scope='notebook'")
        notebook_args = dict(
            notebook=notebook, plan=plan, model=model, max_steps=max_steps, timeout=timeout,
            dry_run=dry_run, symbols=symbols,
        )
        result = await asyncio.wait_for(asyncio.to_thread(execute_plan, **notebook_args), timeout=tool_timeout)
        full_output = plan_result_text(result)
        return mcp_tool_result(
            "execute_plan", arguments, full_output, detail=detail,
            execute_plan=result if isinstance(result, dict) else {},
            history=result.get("history", []) if isinstance(result, dict) else [],
            result_summary=result.get("summary", "") if isinstance(result, dict) else "",
        )
    except TimeoutError as exc:
        message = f"execute_plan exceeded its MCP timeout of {tool_timeout:g}s before Codex's client timeout."
        return _mcp_tool_error_result("execute_plan", arguments, TimeoutError(message), detail=detail)
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("execute_plan", arguments, exc, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("agent_workbench"))
async def agent_workbench_tool(
    goal: str, notebook: str | None = None, contract_file: str | None = None,
    execute: bool = False, max_steps: int = 8, timeout: int = 30,
    tool_timeout: float = 150.0, detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Prepare or execute a taste-aware small-diff workbench run."
    root = await _mcp_workspace_root(ctx)
    notebook = _mcp_workspace_path(notebook, root) if notebook else notebook
    contract_file = _mcp_workspace_path(contract_file, root) if contract_file else contract_file
    max_steps = _mcp_clamp_int(max_steps, 8, 1, 20, "max_steps")
    timeout = _mcp_clamp_int(timeout, 30, 1, 120, "timeout")
    tool_timeout = _mcp_clamp_float(tool_timeout, 150.0, 0.001, 150.0, "tool_timeout")
    arguments = dict(goal=goal, notebook=notebook, contract_file=contract_file, execute=execute, max_steps=max_steps, timeout=timeout, tool_timeout=tool_timeout, detail=detail, workspace_root=str(root) if root else None)
    try:
        result = await asyncio.wait_for(
            asyncio.to_thread(
                agent_workbench_result,
                goal, notebook=notebook, contract_file=contract_file, execute=execute,
                max_steps=max_steps, timeout=timeout,
            ),
            timeout=tool_timeout,
        )
        full_output = result.get("rendered_plan") or result.get("summary", "")
        return mcp_tool_result(
            "agent_workbench", arguments, full_output, detail=detail,
            agent_workbench=result if isinstance(result, dict) else {},
        )
    except TimeoutError as exc:
        message = f"agent_workbench exceeded its MCP timeout of {tool_timeout:g}s before Codex's client timeout."
        return _mcp_tool_error_result("agent_workbench", arguments, TimeoutError(message), detail=detail)
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("agent_workbench", arguments, exc, detail=detail)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("style_check"))
async def style_check_tool(
    path: str = ".", skip_folder_re: str | None = None, skip_path: str | None = None,
    strict: bool = False, delete_after_output: bool = False,
    max_output_chars: int = 12000, max_diagnostics: int = 200, fix: bool = False,
    changed_only: bool = False, ref_a: str | None = "HEAD", ref_b: str | None = None,
    detail: str = "summary", ctx: Context = None,
) -> ToolResult:
    "Print capped chkstyle output, notebook hygiene warnings, private symbol warnings, and global tool usage."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    skip_path = _mcp_workspace_path(skip_path, root) if skip_path else skip_path
    arguments = dict(path=path, skip_folder_re=skip_folder_re, skip_path=skip_path, strict=strict, delete_after_output=delete_after_output, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics, fix=fix, changed_only=changed_only, ref_a=ref_a, ref_b=ref_b, detail=detail, workspace_root=str(root) if root else None)
    try:
        style_output = capture_call(style_check, **{k: v for k, v in arguments.items() if k not in {"detail", "workspace_root"}})
        private_output = _private_symbol_report_text(path)
        full_output = "\n\n".join(chunk for chunk in [private_output, style_output] if chunk)
        report = style_report(
            path, max_output_chars=max_output_chars, max_diagnostics=max_diagnostics,
            changed_only=changed_only, ref_a=ref_a, ref_b=ref_b,
            skip_folder_re=skip_folder_re, skip_path=skip_path,
        )
    except Exception as exc:
        return _mcp_tool_error_result("style_check", arguments, exc, max_output_chars=max_output_chars, detail=detail)
    report["private_symbol_report"] = private_output
    return mcp_tool_result("style_check", arguments, full_output, max_output_chars=max_output_chars, detail=detail, style_report=report)

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("reference"))
async def _reference_tool(
    action: str = "query",
    query: str | None = None,
    top_k: int = 3,
    include_branch: bool = False,
    current_repo: str = ".",
    repos: str | None = None,
    url: str | None = None,
    name: str | None = None,
    version: str = "HEAD",
    package: str | None = None,
    path: str | None = None,
    kind: str | None = None,
    module: str | None = None,
    symbol: str | None = None,
    include_local: bool = True,
    candidate_k: int | None = None,
    explain: bool = True,
    all: bool = False,
    force: bool = False,
    tool_timeout: float = 120.0,
    detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Add, list, ingest, or query reference implementations."
    root = await _mcp_workspace_root(ctx)
    current_repo = _mcp_workspace_path(current_repo, root)
    path = _mcp_workspace_path(path, root) if path else path
    action = str(action or "query").lower()
    top_k = _mcp_clamp_int(top_k, 3, 1, 20, "top_k")
    candidate_k = None if candidate_k is None else _mcp_clamp_int(candidate_k, top_k * 25, top_k, 500, "candidate_k")
    tool_timeout = _mcp_clamp_float(tool_timeout, 120.0, 0.001, 120.0, "tool_timeout")
    arguments = dict(
        action=action, query=query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos,
        url=url, name=name, version=version, package=package, path=path, kind=kind, module=module, symbol=symbol,
        include_local=include_local, candidate_k=candidate_k, explain=explain,
        all=all, force=force, tool_timeout=tool_timeout, detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        if action == "add":
            if not url: raise ValueError("reference action='add' needs url")
            result_data = await asyncio.wait_for(
                asyncio.to_thread(reference_add, url, name=name, version=version, package=package, path=path),
                timeout=tool_timeout,
            )
        elif action == "list":
            result_data = await asyncio.wait_for(asyncio.to_thread(reference_list, path=path), timeout=tool_timeout)
        elif action == "ingest":
            result_data = await asyncio.wait_for(
                asyncio.to_thread(reference_ingest, name=name, all=all, path=path, force=force),
                timeout=tool_timeout,
            )
        elif action == "query":
            if not query: raise ValueError("reference action='query' needs query")
            result_data = await asyncio.wait_for(
                asyncio.to_thread(
                    reference_query,
                    query, top_k=top_k, include_branch=include_branch, current_repo=current_repo, repos=repos, path=path,
                    kind=kind, package=package, module=module, symbol=symbol,
                    include_local=include_local, candidate_k=candidate_k, explain=explain,
                ),
                timeout=tool_timeout,
            )
        else:
            raise ValueError("action must be add, list, ingest, or query")
        full_output = json.dumps(result_data, indent=2, sort_keys=True)
        return mcp_tool_result("reference", arguments, full_output, detail=detail, reference=result_data)
    except TimeoutError as exc:
        message = f"reference exceeded its MCP timeout of {tool_timeout:g}s before Codex's client timeout."
        return _mcp_tool_error_result("reference", arguments, TimeoutError(message), detail=detail)
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("reference", arguments, exc, detail=detail)

In [ ]:
#| hide
import inspect

_reference_params = inspect.signature(_reference_tool).parameters
for _name in ("include_local", "candidate_k", "explain"):
    assert _name in _reference_params

In [ ]:
#| export
@mcp.tool(**_mcp_tool_meta("convert"))
async def convert_tool(
    path: str,
    mode: str = "notebook",
    dest: str | None = None,
    nbs_path: str = "nbs",
    recursive: bool = True,
    maxdepth: int | None = None,
    preserve_tree: bool = True,
    class_lines: int = 100,
    method_lines: int = 10,
    package: str | None = None,
    include: str | None = None,
    exclude: str | None = None,
    skip_init: bool = True,
    include_tests: bool = False,
    force: bool = True,
    run_validation: bool = True,
    dry_run: bool = True,
    detail: str = "summary",
    ctx: Context = None,
) -> ToolResult:
    "Convert Python files/folders or a Python package into nbdev notebooks/project structure."
    root = await _mcp_workspace_root(ctx)
    path = _mcp_workspace_path(path, root)
    dest = _mcp_workspace_path(dest, root) if dest else dest
    nbs_path = _mcp_workspace_path(nbs_path, root)
    arguments = dict(
        path=path, mode=mode, dest=dest, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
        preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package,
        include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, force=force,
        run_validation=run_validation, dry_run=dry_run, detail=detail, workspace_root=str(root) if root else None,
    )
    try:
        full_output = capture_call(
            convert, path=path, mode=mode, dest=dest, nbs_path=nbs_path, recursive=recursive, maxdepth=maxdepth,
            preserve_tree=preserve_tree, class_lines=class_lines, method_lines=method_lines, package=package,
            include=include, exclude=exclude, skip_init=skip_init, include_tests=include_tests, dry_run=dry_run,
            force=force, run_validation=run_validation,
        )
        return mcp_tool_result("convert", arguments, full_output, detail=detail)
    except (Exception, SystemExit) as exc:
        return _mcp_tool_error_result("convert", arguments, exc, detail=detail)

In [ ]:

#| export
def create_mcp():
    "Return the public nbskill FastMCP server."
    return mcp

### Running the server

The CLI entry point only chooses the transport and starts FastMCP. Keeping startup separate from tool registration makes `create_mcp` easy to test without launching a long-running server.

In [ ]:
#| export
def main(
    transport: str = "stdio",  # MCP transport; stdio is what Codex/Claude use for local servers
    show_banner: bool = False,  # Show FastMCP startup banner
):
    "Run the nbskill MCP server."
    _mcp_log_event("server_start", transport=transport, show_banner=show_banner, cwd=str(Path.cwd()), python=sys.executable)
    try:
        mcp.run(transport=transport, show_banner=show_banner)
    except BaseException as exc:
        _mcp_log_exception("server_error", exc, transport=transport)
        raise
    finally:
        _mcp_log_event("server_stop", transport=transport)

### MCP transport-friendly edits

MCP clients should be able to send exact source directly. `source_lines` avoids JSON-string plans, temporary files, and CLI newline decoding entirely, which is useful when the source being written contains escaped notebook text. Unsafe notebook execution is still available through MCP, but it runs in a subprocess so signal-based timeouts stay in a main interpreter.

In [ ]:
mcp_demo = create_mcp()
mcp_tools = {tool.name: tool for tool in await mcp_demo.list_tools()}
assert "edits" in str(mcp_tools["edit_notebook"].parameters)
assert "auto_feedback" in str(mcp_tools["edit_notebook"].parameters)
assert "default_cell_type" in str(mcp_tools["edit_notebook"].parameters)
assert "edit_cell" not in mcp_tools
print("direct MCP schema exposes one structured edit tool")

direct MCP schema exposes one structured edit tool


In [ ]:
with write_demo_notebook("07_mcp_insert_cells.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"')])
    anchor_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="insert_cells", anchor_id=anchor_id, cells=[dict(source_lines=["inserted = True"])])], "auto_feedback": False})
    assert result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[1].source == "inserted = True"
    print("direct MCP edit_notebook inserts structured cells")

direct MCP edit_notebook inserts structured cells


In [ ]:
with write_demo_notebook("07_mcp_edit_notebook.ipynb") as edit_nb:
    nb = new_nb([mk_cell('payload = "old"\nother = 1')])
    edit_cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    edit_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=edit_cell_id, source_lines=[exact_source])], "auto_feedback": False})
    assert edit_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source
    range_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_lines", cell_id=edit_cell_id, start_line=1, end_line=1, replacement_lines=['payload = "range"'])], "auto_feedback": False})
    assert range_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "range"'
    feedback_result = await mcp_demo.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=edit_cell_id, source_lines=["print('mcp feedback')"])]})
    assert "Auto feedback" in feedback_result.structured_content["full_output"]
    assert "mcp feedback" in feedback_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].outputs == []
    print("direct MCP edit_notebook preserves escaped newlines and returns feedback")

direct MCP edit_notebook preserves escaped newlines and returns feedback


In [ ]:
#| hide
old_edit_notebook = globals()["edit_notebook"]


def _slow_mcp_edit_notebook(*args, **kwargs):
    time.sleep(0.05)
    return dict(ok=True, changed=False, no_change=True, dry_run=kwargs.get("dry_run", False), affected_cell_ids=[], text="late result")


try:
    globals()["edit_notebook"] = _slow_mcp_edit_notebook
    timeout_result = await mcp_demo.call_tool(
        "edit_notebook",
        dict(
            path="nbs/data/07_mcp_timeout.ipynb",
            edits=[dict(op="replace_cell", cell_id="abc123", source_lines=["x = 1"])],
            auto_feedback=False,
            tool_timeout=0.01,
        ))
finally: globals()["edit_notebook"] = old_edit_notebook

content = timeout_result.structured_content
health_result = await mcp_demo.call_tool("healthcheck", {})
checks = [
    content["status"] == "failed",
    content["ok"] is False,
    content["error"]["type"] == "TimeoutError",
    "nbskill mcp ok" in health_result.structured_content["full_output"],
]
assert all(checks)
print("MCP edit_notebook timeout returns an error without killing the server")

In [ ]:
with write_demo_notebook("07_mcp_exec_nb.ipynb") as exec_nb_path:
    _write_raw_nb(new_nb([mk_cell("print('mcp exec')")]), exec_nb_path)
    result = await mcp_demo.call_tool("exec_nb", {"path": str(exec_nb_path.resolve()), "allow_new": True, "timeout": 5})
    assert "mcp exec" in result.structured_content["full_output"]
    print("MCP exec_nb ran with focused defaults")

MCP exec_nb ran with focused defaults


In [ ]:
mcp = create_mcp()

In [ ]:
tools = {tool.name: tool for tool in await mcp.list_tools()}

In [ ]:
assert {
    "healthcheck", "doctor", "context", "edit_notebook", "exec_nb", "execute_plan",
    "agent_workbench", "style_check", "reference", "convert",
} <= set(tools)
assert not {"edit_cell", "edit_cell_range", "insert_cells", "apply_notebook_edits"} & set(tools)
assert not {"nb_overview", "nb_chapter", "nb_cell", "show_doc"} & set(tools)

In [ ]:
assert "write_nb" not in tools

In [ ]:
assert "update_cell" not in tools

In [ ]:
assert "batch_edit_nb" not in tools

In [ ]:
assert "execute_project_plan" not in tools

In [ ]:
assert not {
    "project_context", "file_context", "chapter_context", "symbol_context",
    "symbol_graph", "private_symbol_report",
    "py2nb", "py2nbs", "py2nbdev", "new_nbdev_notebook",
    "store_knowledge", "add_behaviour_steering", "get_knowledge",
    "reference_add", "reference_list", "reference_ingest", "reference_query",
} & set(tools)

In [ ]:
assert "read_nb" not in tools

In [ ]:
assert "detail" in str(tools["context"].parameters)
assert "verbose" not in str(tools["context"].parameters)

In [ ]:
assert "show_owner" in str(tools["diff_nb"].parameters)

In [ ]:
assert "cell_id" in str(tools["diff_nb"].parameters)

In [ ]:
assert "after_id" in str(tools["diff_nb"].parameters)

In [ ]:
assert "check_only" in str(tools["exec_nb"].parameters)

In [ ]:
assert "scope" in str(tools["execute_plan"].parameters)
assert "symbols" in str(tools["execute_plan"].parameters)

In [ ]:
assert "notebooks" in str(tools["execute_plan"].parameters)

In [ ]:
assert "edits" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "default_cell_type" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "auto_feedback" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_timeout" in str(tools["edit_notebook"].parameters)
assert "tool_timeout" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "validate_code" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_safe" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "feedback_timeout" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "edits" in str(tools["edit_notebook"].parameters)

In [ ]:
assert "max_output_chars" in str(tools["style_check"].parameters)

In [ ]:
assert "changed_only" in str(tools["style_check"].parameters)

In [ ]:
assert "scopes" in str(tools["doctor"].parameters)

In [ ]:
assert "include_graph" in str(tools["context"].parameters)
assert "mode" in str(tools["convert"].parameters)
assert "action" in str(tools["reference"].parameters)

In [ ]:
assert "delete_after_outout" not in str(tools["style_check"].parameters)

In [ ]:
for name in ("edit_notebook", "execute_plan", "convert"):
    assert "dry_run" in str(tools[name].parameters)

for name in ("style_check", "reference"):
    assert "dry_run" not in str(tools[name].parameters)

In [ ]:
assert "Single notebook-aware reader" in tools["context"].description

In [ ]:
assert "symbol graph payloads" in tools["context"].meta["combine_with"]

In [ ]:
assert {"read", "context", "symbol"} <= set(tools["context"].tags)

In [ ]:
assert {"edit", "notebook", "cell"} <= set(tools["edit_notebook"].tags)

In [ ]:
assert {"edit", "notebook", "text"} <= set(tools["edit_notebook"].tags)

In [ ]:
assert "deterministic notebook edit operations" in tools["edit_notebook"].description

In [ ]:
assert "structured" in tools["edit_notebook"].description

In [ ]:
assert "atomically" in tools["edit_notebook"].description

In [ ]:
assert tools["execute_plan"].meta["feature"] == "agentic_planning"

In [ ]:
assert "Combined former execute_project_plan" in tools["execute_plan"].meta["combine_with"]

In [ ]:
assert tools["agent_workbench"].meta["feature"] == "agentic_planning"

In [ ]:
assert "execute" in str(tools["agent_workbench"].parameters)

In [ ]:
assert tools["convert"].meta["usefulness"] == "situational"

In [ ]:
assert "single-file, folder, and project" in tools["convert"].meta["combine_with"]

In [ ]:
assert tools["reference"].meta["feature"] == "reference_knowledge"

In [ ]:
assert "CLI subprocess" in tools["exec_nb"].description

In [ ]:
#| hide
from contextlib import contextmanager


@contextmanager
def _demo_root(name, mkdir=True):
    root = demo_path(name)
    if mkdir: root.mkdir()
    try:
        yield root
    finally:
        remove_demo_path(root)

In [ ]:
out = StringIO()
with _demo_root("07_mcp_context_graph") as graph_root:
    lib_nb = graph_root / "lib.ipynb"
    call_nb = graph_root / "call.ipynb"
    _write_raw_nb(new_nb([mk_cell("#| export\ndef thing():\n    return 1")]), lib_nb)
    _write_raw_nb(new_nb([mk_cell("from nbskill.lib import thing\nvalue = thing()")]), call_nb)
    context_args = dict(target="thing", scope=str(graph_root))
    with redirect_stdout(out): context_result = await mcp.call_tool("context", context_args)
    graph_payload = context_result.structured_content["context"]["symbol_graphs"][0]
    assert graph_payload["caller_usages"][0]["line"] == "value = thing()"
    assert out.getvalue() == ""
    print("structured context symbol usages:", len(graph_payload["caller_usages"]))

In [ ]:
async def _exercise_mcp_edit_tools(edit_root):
    edit_root.mkdir()
    edit_nb = edit_root / "edit.ipynb"
    nb = new_nb([mk_cell('payload = "old"\nvalue = 1')])
    cell_id = nb.cells[0].id
    _write_raw_nb(nb, edit_nb)
    exact_source = 'payload = "line 1' + chr(92) + 'nline 2"'
    dry_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=cell_id, source_lines=[exact_source])], "auto_feedback": False, "dry_run": True})
    assert dry_result.structured_content["changed"] is True
    assert dry_result.structured_content["dry_run"] is True
    assert dry_result.structured_content["affected_cell_ids"] == [cell_id]
    assert dry_result.structured_content["before_hash"] == dry_result.structured_content["after_hash"]
    assert dry_result.structured_content["planned_hash"] != dry_result.structured_content["before_hash"]
    assert "---" in dry_result.structured_content["full_output"]
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "old"\nvalue = 1'

    edit_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_cell", cell_id=cell_id, source_lines=[exact_source])], "auto_feedback": False})
    assert edit_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == exact_source

    range_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="replace_lines", cell_id=cell_id, start_line=1, end_line=1, replacement_lines=['payload = "range"'])], "auto_feedback": False})
    assert range_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].source == 'payload = "range"'

    insert_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="insert_cells", anchor_id=cell_id, cells=[dict(cell_type="code", source_lines=["inserted = True"])])], "auto_feedback": False})
    assert insert_result.structured_content["changed"] is True
    inserted_id = _read_raw_nb(edit_nb).cells[1].id
    assert _read_raw_nb(edit_nb).cells[1].source == "inserted = True"

    move_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="move_cells", cell_ids=[inserted_id], anchor_id=cell_id, where="before")], "auto_feedback": False})
    assert move_result.structured_content["changed"] is True
    assert _read_raw_nb(edit_nb).cells[0].id == inserted_id

    delete_result = await mcp.call_tool("edit_notebook", {"path": str(edit_nb), "edits": [dict(op="delete_cells", cell_ids=[inserted_id])], "auto_feedback": False})
    assert delete_result.structured_content["changed"] is True

In [ ]:
with _demo_root("07_mcp_edit_tools", mkdir=False) as edit_root:
    await _exercise_mcp_edit_tools(edit_root)
    print("direct MCP edit tools used source_lines without JSON plan text")

In [ ]:
with write_demo_notebook("07_mcp_exec.ipynb") as exec_nb_path:
    _write_raw_nb(new_nb([mk_cell("print('mcp exec')")]), exec_nb_path)
    result = await mcp.call_tool("exec_nb", {"path": str(exec_nb_path.resolve()), "allow_new": True, "timeout": 2})
    assert "mcp exec" in result.structured_content["full_output"]
    print("MCP exec_nb ran with focused defaults")

In [ ]:
#| eval: false
assert _doctor_scope_set("all") == {"error", "warning", "style"}
doctor = _doctor_report(".", scopes="error,warning")
assert doctor["style"] is None
assert "errors" in doctor
assert "warnings" in doctor
assert "status" in doctor

In [ ]:
#| eval: false
style_doctor = _doctor_report(".", scopes="style", max_output_chars=200, max_diagnostics=5)
assert style_doctor["style"] is not None
assert "chkstyle" in style_doctor["style"]

In [ ]:
#| hide
with write_demo_notebook("07_mcp_export_hash_fix.ipynb", base="nbs") as fix_path:
    py_path = None
    try:
        nb = new_nb([
            mk_cell("#| default_exp mcp_hash_fix_demo", cell_type="code"),
            mk_cell("#| export\ndef mcp_hash_fix_demo():\n    return 1", cell_type="code"),
        ])
        _write_raw_nb(nb, fix_path)
        py_path = exported_py_path(fix_path, nb)
        py_path.parent.mkdir(parents=True, exist_ok=True)
        py_path.unlink(missing_ok=True)
        py_path.write_text("def mcp_hash_fix_demo():\n    return 1\n", encoding="utf-8")
        nb = _read_raw_nb(fix_path)
        nb.metadata["nbskill"] = {"exported_py_hash": "stale"}
        _write_raw_nb(nb, fix_path)
        assert any(problem["code"] == "exported-py-hash-mismatch" for problem in notebook_validation_problems(fix_path))
        report = _doctor_report(str(fix_path.resolve()), fix=True, scopes="error,warning")
        assert report["fix"]["applied"]
        assert not any(problem["code"] == "exported-py-hash-mismatch" for problem in notebook_validation_problems(fix_path))
    finally:
        if py_path is not None: py_path.unlink(missing_ok=True)

In [ ]:
#| hide
style_source = "\n".join([
    "import os",
    "import os",
    "",
    "def mcp_style_demo(flag):",
    "",
    "    if flag:",
    "        return 1",
    "    values = [",
    "        1,",
    "        ]",
    "    return values",
])

with write_demo_notebook("07_mcp_style_autofix.ipynb") as style_path:
    _write_raw_nb(new_nb([mk_cell(style_source)]), style_path)
    before = _read_raw_nb(style_path).cells[0].source
    dry_report = _doctor_report(str(style_path), fix=False, scopes="style", max_output_chars=400)
    assert dry_report["style"] is not None
    assert dry_report["fix"]["applied"] == []
    assert _read_raw_nb(style_path).cells[0].source == before

    fixed_report = _doctor_report(str(style_path), fix=True, scopes="style", max_output_chars=400)
    codes = {item["code"] for item in fixed_report["fix"]["applied"]}
    assert {"function-blank-line", "single-line-if", "paren-only-line"} <= codes
    assert "Fixes:" in fixed_report["text"]
    after = _read_raw_nb(style_path).cells[0].source
    assert "if flag: return 1" in after
    assert "1,]" in after
    assert after != before

In [ ]:
#| hide
style_source = "\n".join([
    "def mcp_style_scope_demo(flag):",
    "",
    "    if flag:",
    "        return 1",
])

with write_demo_notebook("07_mcp_style_scope.ipynb") as style_path:
    _write_raw_nb(new_nb([mk_cell(style_source)]), style_path)
    before = _read_raw_nb(style_path).cells[0].source
    report = _doctor_report(str(style_path), fix=True, scopes="error,warning")
    assert all(item.get("code") != "single-line-if" for item in report["fix"]["applied"])
    assert _read_raw_nb(style_path).cells[0].source == before

In [ ]:
long_output = "important header\n" + "x" * 5000
redacted = mcp_tool_result(
    "edit_notebook",
    {"path": "nbs/example.ipynb", "edits": [dict(op="replace_cell", cell_id="abc123", source_lines=["x" * 1000])]},
    long_output,
    edit_notebook={"changed": True, "dry_run": False, "affected_cell_ids": ["abc123"], "diffs": [dict(changed=True)]},
)
summary = redacted.structured_content["summary"]
assert redacted.content[0].text == summary
assert "Result:" not in summary
assert "x" * 200 not in summary
assert redacted.structured_content["full_output"] == redacted.structured_content["preview_output"]
assert redacted.structured_content["call"]["arguments"]["edits"].startswith("<")

huge_structured = mcp_tool_result("context", {"target": "nbs/example.ipynb"}, long_output, context={"cells": [{"source": "y" * 20000}]})
assert len(huge_structured.structured_content["context"]["cells"][0]["source"]) < 13000

full = mcp_tool_result("context", {"target": "nbs/example.ipynb"}, long_output, detail="full")
assert "Result:" in full.structured_content["summary"]
assert "debug" not in full.structured_content

In [ ]:
debug = mcp_tool_result("context", {"target": "nbs/example.ipynb"}, long_output, detail="debug")
assert debug.structured_content["debug"]["arguments"] == {"target": "nbs/example.ipynb"}
assert debug.structured_content["debug"]["raw_output"] == long_output
assert "Result:" in debug.structured_content["summary"]

shape_cases = [
    (
        "context",
        {"target": "foo", "scope": "nbs"},
        "full context",
        {"context": {"resolved_kind": "symbol", "path": "nbs/a.ipynb", "symbol": "foo", "cells": [dict(path="nbs/a.ipynb", cell_id="abc123", source="def foo(): pass")]}},
        ("context completed", "resolved=symbol", "path=nbs/a.ipynb", "symbol=foo", "cells=1", "id=abc123"),
    ),
    (
        "filter_context",
        {"scope": "nbs", "include_re": "foo"},
        "full matches",
        {"filter_context": {"matches": [dict(path="nbs/a.ipynb", cell_id="abc123", cell_idx=7, source="#| export\ndef foo(): pass\nclass Bar:\n    def baz(self): pass")], "total_matches": 3}},
        ("filter_context completed", "matches=1 shown of 3", "id=abc123", "def foo(): pass", "class Bar:", "def baz(self): pass"),
    ),
    (
        "diff_nb",
        {"path": "nbs/a.ipynb"},
        "--- code cell abc123 ---\n-old\n+new\n",
        {},
        ("diff_nb completed", "changed_cells=1", "added_lines=1", "deleted_lines=1", "cells=abc123"),
    ),
    (
        "doctor",
        {"path": "nbs/a.ipynb"},
        "long doctor output",
        {"doctor": {"errors": [dict(code="fatal", path="nbs/a.ipynb", cell_id="abc123", detail="bad")], "warnings": [dict(code="warn", path="nbs/b.ipynb", detail="care")]}},
        ("doctor completed", "errors=1 warnings=1", "fatal:"),
    ),
    (
        "style_check",
        {"path": "nbs/a.ipynb"},
        "long style output",
        {"style_report": {"summary": {"diagnostic_count": 2, "notebook_problem_count": 1, "chkstyle_problem_count": 1}, "problem_chart": {"by_code": {"W1": 2}}, "diagnostics": [dict(code="W1", path="nbs/a.ipynb", detail="style")]}},
        ("style_check completed", "diagnostics=2", "notebook_problems=1", "top_codes=W1=2", "W1:"),
    ),
    (
        "reference",
        {"action": "query", "query": "foo"},
        "{}",
        {"reference": {"backend": "local", "hits": [dict(module="pkg.mod", symbol="thing", dependency_status="direct", score=0.91)]}},
        ("reference completed", "hits=1 backend=local", "pkg.mod.thing: direct score=0.91"),
    ),
    (
        "edit_notebook",
        {"path": "nbs/a.ipynb", "dry_run": True},
        "full diff",
        {"edit_notebook": {"changed": True, "dry_run": True, "affected_cell_ids": ["abc123"], "diffs": [dict(changed=True)]}},
        ("edit_notebook completed", "changed=True", "dry_run=True", "affected_cells=1", "cells=abc123"),
    ),
    (
        "agent_workbench",
        {"goal": "ship"},
        "large context payload",
        {"agent_workbench": {"contract": {"goal": "ship"}, "context": {"selected_notebooks": [dict(path="nbs/a.ipynb")]}, "expected_gates": {"hard": ["exec_nb"]}}},
        ("agent_workbench completed", "goal=ship", "notebooks=nbs/a.ipynb", "gates=exec_nb"),
    ),
]

old_doctor_warnings = globals().get("_doctor_warnings")
try:
    globals()["_doctor_warnings"] = lambda *args, **kwargs: []
    for tool, arguments, raw, structured, fragments in shape_cases:
        result = mcp_tool_result(tool, arguments, raw, **structured)
        summary = result.structured_content["summary"]
        assert "Result:" not in summary
        assert result.structured_content["full_output"] == raw
        for fragment in fragments:
            assert fragment in summary, (tool, fragment, summary)
finally:
    globals()["_doctor_warnings"] = old_doctor_warnings

In [ ]:
calls = []

In [ ]:
old_doctor_warnings = _doctor_warnings

In [ ]:
def _capture_mcp_warning_scopes():
    try:
        def fake_doctor_warnings(path=".", scope_path=None, scope_cell_ids=None):
            cell_scope = None if scope_cell_ids is None else set(scope_cell_ids)
            calls.append((path, scope_path, cell_scope))
            return []

        globals()["_doctor_warnings"] = fake_doctor_warnings
        mcp_tool_result("healthcheck", {}, "ok")
        mcp_tool_result("context", {"target": "nbs"}, "Project context")
        mcp_tool_result("context", {"target": "nbs/chapter.ipynb"}, "Cell id=ch1\nCell id=ch2")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "diff without cell ids")
        mcp_tool_result("diff_nb", {"path": "nbs/07_mcp.ipynb"}, "--- code cell abc123 ---\nchanged")
        mcp_tool_result("edit_notebook", {"path": "nbs/update_scope.ipynb", "edits": [dict(op="replace_cell", cell_id="upd123", source_lines=["x"])]}, "edit_notebook changed")
        mcp_tool_result("edit_notebook", {"path": "nbs/batch_a.ipynb", "edits": [dict(op="replace_cell", path="nbs/batch_b.ipynb", cell_id="b1", source_lines=["b"])]}, "edit_notebook changed")
    finally:
        globals()["_doctor_warnings"] = old_doctor_warnings

In [ ]:
_capture_mcp_warning_scopes()

In [ ]:
assert calls == [
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", set()),
    ("nbs/07_mcp.ipynb", "nbs/07_mcp.ipynb", {"abc123"}),
]


In [ ]:
sample = [
    _warning("other", "other notebook", path="nbs/other.ipynb"),
    _warning("same_other_cell", "same notebook other cell", path="nbs/07_mcp.ipynb", cell_id="other-cell"),
    _warning("current", "current displayed cell", path="nbs/07_mcp.ipynb", cell_id="abc123"),
    _warning("notebook_export_missing", "current notebook", path="nbs/07_mcp.ipynb"),
]

In [ ]:
filtered = _filter_warnings_for_scope(sample, Path(".").resolve(), "nbs/07_mcp.ipynb", {"abc123"})
assert [item["code"] for item in filtered] == ["current", "notebook_export_missing"]

In [ ]:
assert not _cell_export_relevant({"cell_type": "code", "source": ["print('demo')"]})
assert _cell_export_relevant({"cell_type": "code", "source": ["#| export\ndef exported():\n    pass"]})
module_path = Path("nbskill/mcp.py")
if not module_path.exists(): module_path = Path("../nbskill/mcp.py")
owner = generated_owner(module_path)
assert owner and owner.name == "07_mcp.ipynb"
assert "07_mcp.ipynb" in _owner_output(module_path)
assert git_root("nbs/07_mcp.ipynb") == git_root(".")
assert _rel_to_root("nbs/07_mcp.ipynb", git_root(".")) == "nbs/07_mcp.ipynb"

In [ ]:
with write_demo_notebook("07_mcp_insert_lines.ipynb") as path:
    nb = new_nb([mk_cell("anchor = True")])
    anchor_id = nb.cells[0].id
    _write_raw_nb(nb, path)
    mcp = _mcp_mod.create_mcp()
    result = await mcp.call_tool(
        "edit_notebook",
        {
            "path": str(path),
            "edits": [dict(op="insert_cells", anchor_id=anchor_id, cells=[dict(cell_type="code", source_lines=['source = "line 1\\nline 2"'])])],
            "auto_feedback": False,
        },
    )
    assert result.structured_content["changed"] is True
    assert _read_raw_nb(path).cells[1].source == 'source = "line 1\\nline 2"'

In [ ]:
with write_demo_notebook("07_mcp_sample.ipynb") as path:
    _example_write_nb(
        str(path),
        "%%code\n"
        "#| default_exp sample\n"
        "def sample():\n"
        "    return 'ok'",
        replace=True,
    )
    text = capture_notebook_call(_example_context, path, target=str(path))
    assert "def sample():" in text
    print("captured file context lines:", len(text.splitlines()))

In [ ]:
assert as_text(None) == ""
assert as_text({"ok": True}) == "{'ok': True}"
assert capture_call(lambda: "returned") == "returned"

In [ ]:
def _prints_and_returns():
    print("printed")
    return "returned"

In [ ]:
assert capture_call(_prints_and_returns) == "printed"

In [ ]:
def _prints_and_exits():
    print("before exit")
    raise SystemExit(7)

In [ ]:
try:
    capture_call(_prints_and_exits)
except RuntimeError as exc:
    assert "before exit" in str(exc)
    assert "SystemExit: 7" in str(exc)
else:
    raise AssertionError("SystemExit should be converted to RuntimeError for MCP tools")

In [ ]:
import time

In [ ]:
original_stdout = sys.stdout
outputs = []
errors = []
entered = threading.Event()

In [ ]:
def _slow_print(label, delay, signal=None):
    def inner():
        if signal is not None: signal.set()
        time.sleep(delay)
        print(label)
    return inner

In [ ]:
def _capture_worker(label, delay, signal=None):
    try:
        outputs.append(capture_call(_slow_print(label, delay, signal=signal)))
    except BaseException as exc:
        errors.append(exc)

In [ ]:
cell_stdout = sys.stdout
threads = [
    threading.Thread(target=_capture_worker, args=("first", 0.03, entered)),
    threading.Thread(target=_capture_worker, args=("second", 0.01)),
]
threads[0].start()
assert entered.wait(1)
threads[1].start()
for thread in threads: thread.join()

assert errors == []
assert sorted(outputs) == ["first", "second"]
assert sys.stdout is cell_stdout